# Grainology State and Mandi Price Forecaster

This standalone Kaggle notebook is built from the working Grainology notebook and preserves its live-price sanity checks, Supabase merge, transparent visual diagnostics, and website release contract.

The model path is upgraded with:

1. Horizon-embargoed chronological train, calibration, and final holdout windows
2. Train-only feature imputation and calibration-only ensemble selection
3. Temporal-fold promotion gates against persistence
4. Split-conformal prediction intervals with measured holdout coverage
5. State-aware and mandi-aware forecasts for 7, 30, and 90 days
6. Evaluation and data-drift reports included in every release

The website-facing state files remain backward compatible. Optional mandi files are added by the generated mandi notebook.


In [ ]:
%pip install -q "catboost>=1.2" "lightgbm>=4.0" "xgboost>=2.0" "optuna>=3.6" "jsonschema>=4.21" "matplotlib>=3.8" "numpy>=1.26" "pandas>=2.2" "polars>=1.0" "pyarrow>=15" "requests>=2.31" "scikit-learn>=1.4" "supabase>=2.0"


## Runtime Notes

This notebook was generated with the **showcase** profile.

- **showcase** is the production default: audited temporal evaluation, fixed model candidates, calibrated intervals, and fast all-mandi sidecars. It disables Optuna and full per-mandi retraining so scheduled runs finish reliably.
- **research** enables Optuna and full global mandi-aware retraining. Use it for manual benchmark experiments, not the daily release job.

Active defaults in this build:

- `ENABLE_OPTUNA_TUNING=false`
- `OPTUNA_TRIALS=5`
- `OPTUNA_TIMEOUT_SECONDS=30`
- `MAX_TRAIN_ROWS_PER_MODEL=250000`
- `TEMPORAL_VALIDATION_FOLDS=3`
- `MIN_TEMPORAL_FOLD_WIN_RATIO=0.50`
- `EVALUATION_HOLDOUT_DAYS=365`
- `ENSEMBLE_CALIBRATION_DAYS=365`
- `MAX_BIAS_CORRECTION_PCT=8`
- `CONFORMAL_ALPHA=0.10`

Reported MAPE and MAE come from an untouched chronological holdout. Do not compare them with the older random-split dashboard score. The release contract remains compatible with the Grainology website.


## Source: config

In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def env_bool(name: str, default: bool = False) -> bool:
    return str(os.environ.get(name, str(default))).strip().lower() in {"1", "true", "yes", "y"}


INPUT_ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
STAGING_DIR = WORK_ROOT / "release_staging"
RELEASE_DIR = WORK_ROOT / "release"
MODEL_DIR = STAGING_DIR / "models"

CANONICAL_PARQUET = STAGING_DIR / "canonical_daily.parquet"
CANONICAL_CSV = STAGING_DIR / "canonical_daily.csv"

TARGET_GRAINS = ["Wheat", "Paddy", "Maize", "Mustard"]
HORIZONS = [7, 30, 90]
CANONICAL_SCHEMA_VERSION = "2.1"
RELEASE_SCHEMA_VERSION = "2.0"
MODEL_MODE = "temporal_ensemble_state_mandi_v4"
NOTEBOOK_RUN_PROFILE = os.environ.get("GRAINOLOGY_RUN_PROFILE", "showcase").strip().lower()

AGGREGATION_METHOD = os.environ.get("AGGREGATION_METHOD", "median").strip().lower() or "median"
FORCE_FULL_REBUILD = env_bool("FORCE_FULL_REBUILD", False)
FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)
AI_PREDICTION_BUCKET = os.environ.get("AI_PREDICTION_BUCKET", "ai-predictions")

# Training and validation controls.
MIN_STATE_OBSERVED_DAYS = int(os.environ.get("MIN_STATE_OBSERVED_DAYS", "240"))
MIN_VALIDATION_SAMPLES = int(os.environ.get("MIN_VALIDATION_SAMPLES", "20"))
TARGET_MATCH_TOLERANCE_DAYS = int(os.environ.get("TARGET_MATCH_TOLERANCE_DAYS", "3"))
VALIDATION_FOLDS = int(os.environ.get("VALIDATION_FOLDS", "4"))
BOOTSTRAP_ITERATIONS = int(os.environ.get("BOOTSTRAP_ITERATIONS", "200"))
GATE_CONFIDENCE_LEVEL = float(os.environ.get("GATE_CONFIDENCE_LEVEL", "0.90"))
MIN_MAPE_IMPROVEMENT = float(os.environ.get("MIN_MAPE_IMPROVEMENT", "0.10"))
MIN_RELATIVE_MAPE_IMPROVEMENT = float(os.environ.get("MIN_RELATIVE_MAPE_IMPROVEMENT", "0.02"))
CONFORMAL_ALPHA = float(os.environ.get("CONFORMAL_ALPHA", "0.10"))
# Leakage-safe evaluation and calibrated uncertainty controls.
TEMPORAL_VALIDATION_FOLDS = int(os.environ.get("TEMPORAL_VALIDATION_FOLDS", "3"))
MIN_TEMPORAL_FOLD_WIN_RATIO = float(os.environ.get("MIN_TEMPORAL_FOLD_WIN_RATIO", "0.50"))
EVALUATION_HOLDOUT_RATIO = float(os.environ.get("EVALUATION_HOLDOUT_RATIO", "0.20"))
ENSEMBLE_CALIBRATION_RATIO = float(os.environ.get("ENSEMBLE_CALIBRATION_RATIO", "0.35"))
EVALUATION_HOLDOUT_DAYS = int(os.environ.get("EVALUATION_HOLDOUT_DAYS", "365"))
ENSEMBLE_CALIBRATION_DAYS = int(os.environ.get("ENSEMBLE_CALIBRATION_DAYS", "365"))
MAX_BIAS_CORRECTION_PCT = float(os.environ.get("MAX_BIAS_CORRECTION_PCT", "8"))
REFIT_SELECTED_MODELS_ON_FULL_DATA = env_bool("REFIT_SELECTED_MODELS_ON_FULL_DATA", True)


# Compatibility controls retained for the existing prediction schema.
# Model selection and reported metrics always use chronological holdout windows.
VALIDATION_STRATEGY = "horizon_embargo_temporal_holdout"
VALIDATION_FRACTION = float(os.environ.get("VALIDATION_FRACTION", "0.10"))
DASHBOARD_STYLE_GATING = env_bool("DASHBOARD_STYLE_GATING", True)
DASHBOARD_ENSEMBLE_PREFERENCE_MARGIN = float(os.environ.get("DASHBOARD_ENSEMBLE_PREFERENCE_MARGIN", "1.02"))
ACCURACY_TARGET_MAPE = float(os.environ.get("ACCURACY_TARGET_MAPE", "2.0"))
TRAINING_SCOPE = os.environ.get("TRAINING_SCOPE", "all").strip().lower()
NATIONAL_PARITY_FORECASTS = env_bool("NATIONAL_PARITY_FORECASTS", True)
NATIONAL_DAILY_FILL = env_bool("NATIONAL_DAILY_FILL", True)
NATIONAL_FFILL_LIMIT_DAYS = int(os.environ.get("NATIONAL_FFILL_LIMIT_DAYS", "14"))
INCLUDE_SKLEARN_FALLBACK_MODELS = env_bool("INCLUDE_SKLEARN_FALLBACK_MODELS", False)

# Runtime / model controls.
FORECAST_HISTORY_DAYS = int(os.environ.get("FORECAST_HISTORY_DAYS", "365"))
EFFICIENCY_MAX_ROWS_PER_SERIES = int(os.environ.get("EFFICIENCY_MAX_ROWS_PER_SERIES", "0"))  # 0 = keep full history
MAX_TRAIN_ROWS_PER_MODEL = int(os.environ.get("MAX_TRAIN_ROWS_PER_MODEL", "250000"))
ENSEMBLE_PRUNE_RATIO = float(os.environ.get("ENSEMBLE_PRUNE_RATIO", "1.50"))
ENABLE_OPTUNA_TUNING = env_bool("ENABLE_OPTUNA_TUNING", False)
OPTUNA_MODELS = [name.strip() for name in os.environ.get(
    "OPTUNA_MODELS", "catboost,lightgbm,xgboost"
).split(",") if name.strip()]
OPTUNA_TRIALS = int(os.environ.get("OPTUNA_TRIALS", "5"))
OPTUNA_TIMEOUT_SECONDS = int(os.environ.get("OPTUNA_TIMEOUT_SECONDS", "30"))

# Horizon-aware safety rails. These are not uncertainty intervals.
HORIZON_PRICE_CLIP_BOUNDS = {
    7: (0.70, 1.40),
    30: (0.55, 1.75),
    90: (0.40, 2.20),
}

# Broad, grain-specific cleaning bounds. Mustard is intentionally wider than the old 20,000 cap.
GRAIN_PRICE_BOUNDS = {
    "Wheat": (100.0, 25000.0),
    "Paddy": (100.0, 25000.0),
    "Maize": (100.0, 25000.0),
    "Mustard": (100.0, 40000.0),
}

# Release quality gates.
MAX_DATA_STALENESS_DAYS = int(os.environ.get("MAX_DATA_STALENESS_DAYS", "7"))
MAX_STATE_ACTUAL_STALENESS_DAYS = int(os.environ.get("MAX_STATE_ACTUAL_STALENESS_DAYS", "7"))
MAX_EFFICIENCY_FILE_MB = float(os.environ.get("MAX_EFFICIENCY_FILE_MB", "160"))
MIN_ML_SLOT_SHARE = float(os.environ.get("MIN_ML_SLOT_SHARE", "0.01"))
FAIL_ON_QUALITY_GATE = env_bool("FAIL_ON_QUALITY_GATE", True)
MAX_FORECAST_CHANGE_PCT = {7: 40.0, 30: 75.0, 90: 140.0}

REQUIRED_RELEASE_FILES = [
    "manifest.json",
    "predictions.json",
    "actuals.json",
    "forecast_series.json",
    "historical_efficiency.json",
    "backtest.json",
    "reasoning.json",
    "states.json",
    "metrics.json",
    "canonical_daily.parquet",
    "canonical_daily.csv",
    "checksums.json",
]

# Transparency controls. These affect notebook reporting only, not the website schema.
TRANSPARENT_MODE = env_bool("TRANSPARENT_MODE", True)
TRANSPARENCY_PREVIEW_ROWS = int(os.environ.get("TRANSPARENCY_PREVIEW_ROWS", "12"))
TRANSPARENCY_MAX_SCHEMA_FILES = int(os.environ.get("TRANSPARENCY_MAX_SCHEMA_FILES", "8"))
TRANSPARENCY_VALIDATION_PLOT_TAIL = int(os.environ.get("TRANSPARENCY_VALIDATION_PLOT_TAIL", "180"))
TRANSPARENCY_TOP_STATES = int(os.environ.get("TRANSPARENCY_TOP_STATES", "20"))







## 0. Live Run Control Centre

This cell executes near the beginning so the notebook immediately shows what it is going to do, which settings are active, and where outputs will be written.

In [ ]:
import platform
import sys
import pandas as pd
from IPython.display import display

run_configuration = pd.DataFrame([{
    "notebook_profile": NOTEBOOK_RUN_PROFILE,
    "transparent_mode": TRANSPARENT_MODE,
    "input_root": str(INPUT_ROOT),
    "working_root": str(WORK_ROOT),
    "release_directory": str(RELEASE_DIR),
    "target_grains": ", ".join(TARGET_GRAINS),
    "horizons_days": ", ".join(map(str, HORIZONS)),
    "aggregation": AGGREGATION_METHOD,
    "force_full_rebuild": FORCE_FULL_REBUILD,
    "force_retrain": FORCE_RETRAIN,
    "minimum_state_days_for_serving": MIN_STATE_OBSERVED_DAYS,
    "minimum_validation_samples": MIN_VALIDATION_SAMPLES,
    "target_tolerance_days": TARGET_MATCH_TOLERANCE_DAYS,
    "validation_folds": VALIDATION_FOLDS,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "gate_confidence": GATE_CONFIDENCE_LEVEL,
    "conformal_coverage_target": 1 - CONFORMAL_ALPHA,
    "max_train_rows_per_model": MAX_TRAIN_ROWS_PER_MODEL,
    "optuna_enabled": ENABLE_OPTUNA_TUNING,
    "optuna_models": ", ".join(OPTUNA_MODELS),
    "optuna_trials_per_model": OPTUNA_TRIALS,
    "efficiency_rows_per_series": EFFICIENCY_MAX_ROWS_PER_SERIES,
    "fail_on_quality_gate": FAIL_ON_QUALITY_GATE,
}])

display(run_configuration)
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()}")
print("\nExecution map:")
print("  Inputs -> previous snapshot -> historical normalization -> Supabase delta")
print("  -> priority merge -> data audit -> corrected feature engineering")
print("  -> tolerance-aligned labels -> embargoed training -> robust model gate")
print("  -> calibrated forecasts -> true holdout efficiency -> reasoning -> quality-gated release")


## Source: historical_loader

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


COLUMN_ALIASES = {
    "date": ["date", "arrival_date", "price_date", "reported_date"],
    "state_name": ["state", "state_name", "state_name_en"],
    "district": ["district", "district_name"],
    "market": ["market", "market_name"],
    "grain": ["commodity", "cmdt_name", "commodity_name", "grain"],
    "variety": ["variety", "variety_name"],
    "grade": ["grade"],
    "price": ["modal_price", "modalprice", "price", "modal_price_rs_quintal"],
    "price_low": ["min_price", "minprice", "price_low"],
    "price_high": ["max_price", "maxprice", "price_high"],
    "arrival": ["arrival", "arrival_qty", "arrival_metric_tonnes", "arrival_mt"],
}


def normalize_grain(value: object) -> str | None:
    text = str(value or "").strip().lower()
    if not text:
        return None
    if "mustard oil" in text or ("oil" in text and "mustard" in text):
        return None
    if "wheat" in text:
        return "Wheat"
    if "paddy" in text:
        return "Paddy"
    if "maize" in text or "corn" in text:
        return "Maize"
    if "mustard" in text or "sarson" in text or "rapeseed" in text or "rape seed" in text:
        return "Mustard"
    return None


def discover_data_files(root: Path = INPUT_ROOT) -> list[Path]:
    parquet_files = sorted(path for path in root.rglob("*.parquet") if path.is_file())
    csv_files = sorted(path for path in root.rglob("*.csv") if path.is_file())
    yearly_parquets = [path for path in parquet_files if path.stem.isdigit()]
    latest_csvs = [path for path in csv_files if path.name.lower() == "latest_data.csv"]
    yearly_csvs = [path for path in csv_files if path.stem.isdigit()]
    if yearly_parquets:
        return yearly_parquets + latest_csvs
    if yearly_csvs:
        return yearly_csvs + latest_csvs
    if parquet_files:
        return parquet_files + latest_csvs
    return csv_files


def resolve_columns(columns: Iterable[str]) -> dict[str, str | None]:
    by_lower = {column.lower().strip(): column for column in columns}
    return {
        target: next((by_lower[alias] for alias in aliases if alias in by_lower), None)
        for target, aliases in COLUMN_ALIASES.items()
    }


def normalize_frame(df: pd.DataFrame, source: str, source_priority: int) -> pd.DataFrame:
    resolved = resolve_columns(df.columns)
    required = ["date", "state_name", "grain", "price"]
    if any(resolved[column] is None for column in required):
        return pd.DataFrame()

    out = pd.DataFrame()
    for column, source_column in resolved.items():
        out[column] = np.nan if source_column is None else df[source_column]

    out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=True).dt.date.astype("string")
    out["state_name"] = out["state_name"].astype("string").str.strip()
    out["grain"] = out["grain"].map(normalize_grain)
    out["variety"] = out["variety"].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA})
    out["grade"] = out["grade"].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA})
    for column in ["price", "price_low", "price_high", "arrival"]:
        out[column] = pd.to_numeric(out[column], errors="coerce")

    valid_price = pd.Series(False, index=out.index)
    for grain, (lower, upper) in GRAIN_PRICE_BOUNDS.items():
        valid_price |= out["grain"].eq(grain) & out["price"].between(lower, upper)

    out = out[
        out["date"].notna()
        & out["state_name"].notna()
        & out["grain"].isin(TARGET_GRAINS)
        & valid_price
    ].copy()

    out["market_count"] = 1
    out["state_id"] = pd.NA
    out["state_key"] = out["state_name"].str.lower().str.replace(r"[^a-z0-9]+", "-", regex=True).str.strip("-")
    out["is_observed"] = True
    out["source"] = source
    out["source_priority"] = source_priority
    out["source_fetched_at"] = pd.NaT
    return out


def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    price_agg = "mean" if AGGREGATION_METHOD == "mean" else "median"
    grouped = df.groupby(["date", "state_name", "grain"], as_index=False, observed=True).agg(
        state_id=("state_id", "first"),
        state_key=("state_key", "first"),
        price=("price", price_agg),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_count", "sum"),
        variety_count=("variety", "nunique"),
        grade_count=("grade", "nunique"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )

    all_states = df.groupby(["date", "grain"], as_index=False, observed=True).agg(
        price=("price", price_agg),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_count", "sum"),
        variety_count=("variety", "nunique"),
        grade_count=("grade", "nunique"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )
    all_states["state_name"] = "All States"
    all_states["state_id"] = "100006"
    all_states["state_key"] = "all-states"
    return pd.concat([grouped, all_states[grouped.columns]], ignore_index=True)


def load_path(path: Path, source_priority: int) -> pd.DataFrame:
    try:
        df = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    except Exception as exc:
        print(f"Skipping {path}: {exc}")
        return pd.DataFrame()
    source = "latest_csv" if path.name.lower() == "latest_data.csv" else "historical"
    return normalize_frame(df, source=source, source_priority=source_priority)


def load_historical_sources() -> pd.DataFrame:
    files = discover_data_files()
    if not files:
        print(f"No historical files found under {INPUT_ROOT}")
        return pd.DataFrame()
    frames = []
    for path in files:
        priority = 2 if path.name.lower() == "latest_data.csv" else 1
        frame = load_path(path, priority)
        if not frame.empty:
            frames.append(frame)
    if not frames:
        return pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    daily = aggregate_daily(raw)
    print(f"Loaded {len(daily):,} canonical historical rows from {len(files)} files")
    return daily


## Source: supabase_source

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from supabase import create_client


def read_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value.strip()
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None


def get_supabase_client():
    url = read_secret("SUPABASE_URL")
    key = read_secret("SUPABASE_SECRET_KEY") or read_secret("SUPABASE_SERVICE_ROLE_KEY")
    if not url or not key:
        print("Supabase secrets are missing; continuing without live actual deltas")
        return None
    return create_client(url, key)


def get_supabase_actuals_watermark(client) -> str | None:
    try:
        response = (
            client.table("agmarknet_ai_actuals")
            .select("date")
            .order("date", desc=True)
            .limit(1)
            .execute()
        )
        rows = response.data or []
        return rows[0].get("date") if rows else None
    except Exception as exc:
        print(f"Could not read Supabase actuals watermark: {exc}")
        return None


def download_previous_canonical(destination: Path) -> bool:
    client = get_supabase_client()
    if not client:
        return False
    for object_path in [
        "canonical/latest/canonical_daily.parquet",
        "canonical/latest/canonical_daily.csv",
    ]:
        try:
            payload = client.storage.from_(AI_PREDICTION_BUCKET).download(object_path)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload if isinstance(payload, bytes) else payload.read())
            print(f"Downloaded previous canonical snapshot: {object_path}")
            return True
        except Exception as exc:
            print(f"Previous canonical download skipped for {object_path}: {exc}")
    return False


def fetch_supabase_actuals(canonical_latest_date: str | None) -> pd.DataFrame:
    client = get_supabase_client()
    if not client:
        return pd.DataFrame()

    watermark = get_supabase_actuals_watermark(client)
    print(f"Supabase agmarknet_ai_actuals latest date: {watermark or 'none'}")
    print(f"Fetching Supabase actual rows newer than canonical date: {canonical_latest_date or 'none'}")

    rows = []
    page_size = 1000
    offset = 0
    while True:
        query = (
            client.table("agmarknet_ai_actuals")
            .select("*")
            .order("date", desc=False)
            .range(offset, offset + page_size - 1)
        )
        if canonical_latest_date:
            query = query.gt("date", str(canonical_latest_date))
        response = query.execute()
        batch = response.data or []
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size

    if not rows:
        print("Fetched 0 Supabase actual rows")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = df[df["grain"].isin(TARGET_GRAINS)].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    for column in ["price_low", "price_high", "arrival"]:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    if "market_count" in df.columns:
        df["market_count"] = pd.to_numeric(df["market_count"], errors="coerce").fillna(0).astype(int)
    df = df[df["price"].gt(0)]
    df["is_observed"] = True
    df["source"] = "supabase_agmarknet"
    df["source_priority"] = 3

    if df.empty:
        print("Fetched Supabase rows, but none survived grain/date/price filtering")
        return df

    print(
        f"Fetched {len(df):,} Supabase actual rows "
        f"from {df['date'].min()} to {df['date'].max()}"
    )
    print(
        "Supabase rows by grain:",
        df.groupby("grain").size().sort_index().to_dict(),
    )
    print(
        "Supabase rows by state:",
        df.groupby("state_name").size().sort_index().to_dict(),
    )
    return df


## Source: canonical_dataset

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd


CANONICAL_COLUMNS = [
    "date", "state_name", "state_id", "state_key", "grain", "price", "price_low", "price_high",
    "arrival", "market_count", "variety_count", "grade_count", "is_observed", "source",
    "source_priority", "source_fetched_at",
]


def align_canonical(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)
    out = df.copy()
    for column in CANONICAL_COLUMNS:
        if column not in out.columns:
            out[column] = pd.NA
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.date.astype("string")
    for column in ["price", "price_low", "price_high", "arrival", "variety_count", "grade_count"]:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    out["source_priority"] = pd.to_numeric(out["source_priority"], errors="coerce").fillna(1).astype(int)
    out["market_count"] = pd.to_numeric(out["market_count"], errors="coerce").fillna(0).astype(int)
    out["variety_count"] = out["variety_count"].fillna(0).astype(int)
    out["grade_count"] = out["grade_count"].fillna(0).astype(int)
    out["is_observed"] = out["is_observed"].fillna(True).astype(bool)
    return out[CANONICAL_COLUMNS].dropna(subset=["date", "state_name", "grain", "price"])


def load_previous_canonical() -> pd.DataFrame:
    previous = STAGING_DIR / "previous_canonical.parquet"
    if FORCE_FULL_REBUILD:
        return pd.DataFrame()
    if not previous.exists():
        download_previous_canonical(previous)
    if previous.exists():
        try:
            return align_canonical(pd.read_parquet(previous))
        except Exception:
            return align_canonical(pd.read_csv(previous))
    return pd.DataFrame()


def merge_priority(frames: list[pd.DataFrame]) -> pd.DataFrame:
    aligned = [align_canonical(frame) for frame in frames if frame is not None and not frame.empty]
    if not aligned:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)
    merged = pd.concat(aligned, ignore_index=True)
    merged = merged.sort_values(["date", "state_name", "grain", "source_priority"])
    merged = merged.drop_duplicates(["date", "state_name", "grain"], keep="last")
    return merged.sort_values(["grain", "state_name", "date"]).reset_index(drop=True)


def validate_canonical(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("Canonical dataset is empty")
    duplicate_count = df.duplicated(["date", "state_name", "grain"]).sum()
    if duplicate_count:
        raise ValueError(f"Canonical dataset has {duplicate_count} duplicate keys")
    if df["price"].le(0).any():
        raise ValueError("Canonical dataset contains nonpositive prices")
    if "All States" not in set(df["state_name"]):
        raise ValueError("Canonical dataset does not contain All States rows")


def save_canonical(df: pd.DataFrame) -> None:
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(CANONICAL_PARQUET, index=False)
    df.to_csv(CANONICAL_CSV, index=False)
    shutil.copy(CANONICAL_PARQUET, Path("/kaggle/working/canonical_daily.parquet"))
    shutil.copy(CANONICAL_CSV, Path("/kaggle/working/canonical_daily.csv"))


def build_canonical_dataset() -> pd.DataFrame:
    previous = load_previous_canonical()
    latest_date = None if previous.empty else str(previous["date"].max())
    historical = pd.DataFrame() if not previous.empty and not FORCE_FULL_REBUILD else load_historical_sources()
    supabase_delta = fetch_supabase_actuals(latest_date)
    canonical = merge_priority([historical, previous, supabase_delta])
    validate_canonical(canonical)
    save_canonical(canonical)
    print(f"Canonical rows: {len(canonical):,}; latest date: {canonical['date'].max()}")
    return canonical


## Source: train

In [ ]:
from __future__ import annotations

import math
import pickle
import time

import numpy as np
import pandas as pd

try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

PRICE_LAGS = [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 180, 365]
DASHBOARD_PRICE_LAGS = [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90]
ROLL_WINDOWS = [7, 14, 30, 60, 90, 180]
EWMA_SPANS = [7, 14, 30, 90, 180]
LOG_RETURN_LAGS = [1, 7, 14, 30]
TREND_WINDOWS = [7, 30, 365]
FOURIER_PERIODS = [7, 30, 90, 365]
FOURIER_K = 4
ARRIVAL_LAGS = [1, 2, 3, 7, 14, 30, 60, 90]
ARRIVAL_ROLLS = [7, 14, 30, 60, 90]
CROSS_GRAIN_LAGS = [7, 14, 30]

FEATURE_COLUMNS = []
FEATURE_COLUMNS += [f"price_lag_{lag}" for lag in PRICE_LAGS]
FEATURE_COLUMNS += [f"log_price_lag_{lag}" for lag in DASHBOARD_PRICE_LAGS]
FEATURE_COLUMNS += [f"return_{period}" for period in [1, 3, 7, 14, 30, 90]]
FEATURE_COLUMNS += [f"log_ret_lag_{lag}" for lag in LOG_RETURN_LAGS]
for window in ROLL_WINDOWS:
    FEATURE_COLUMNS += [
        f"rolling_mean_{window}", f"rolling_median_{window}", f"rolling_min_{window}",
        f"rolling_max_{window}", f"rolling_std_{window}", f"rolling_range_{window}",
        f"rolling_cv_{window}", f"price_pct_range_{window}", f"vol_log_ret_{window}",
    ]
FEATURE_COLUMNS += [f"ewm_mean_{span}" for span in EWMA_SPANS]
FEATURE_COLUMNS += [f"ewma_ratio_{span}" for span in EWMA_SPANS]
FEATURE_COLUMNS += [f"trend_slope_{window}" for window in TREND_WINDOWS]
FEATURE_COLUMNS += [
    "momentum_7_30", "momentum_30_90", "price_vs_mean_30", "price_vs_mean_90",
    "price_range_pct", "price_spread_lag1", "price_spread_roll7",
]
FEATURE_COLUMNS += ["arrival"]
FEATURE_COLUMNS += [f"arrival_lag_{lag}" for lag in ARRIVAL_LAGS]
FEATURE_COLUMNS += [f"log_arr_lag_{lag}" for lag in ARRIVAL_LAGS]
for window in ARRIVAL_ROLLS:
    FEATURE_COLUMNS += [f"arrival_rolling_mean_{window}", f"arrival_rolling_std_{window}"]
FEATURE_COLUMNS += ["arrival_trend_7", "arrival_trend_30", "arrival_stale"]
FEATURE_COLUMNS += [
    "market_count", "market_count_lag_1", "market_count_lag_7",
    "market_count_rolling_mean_7", "market_count_rolling_mean_30",
    "variety_count", "grade_count",
]
FEATURE_COLUMNS += [
    "month", "quarter", "day_of_week", "day_of_year", "week_of_year", "year_index",
    "season_sin", "season_cos", "month_sin", "month_cos", "is_monsoon",
    "is_harvest_rabi", "is_harvest_kharif", "is_harvest_season", "is_sowing_season",
    "months_to_harvest",
]
for period in FOURIER_PERIODS:
    for k in range(1, FOURIER_K + 1):
        FEATURE_COLUMNS += [f"sin_{period}d_k{k}", f"cos_{period}d_k{k}"]
FEATURE_COLUMNS += ["msp", "msp_gap_ratio", "price_above_msp", "msp_pct_diff"]
FEATURE_COLUMNS += [
    "national_price", "national_lag_1", "national_lag_7", "national_lag_30",
    "national_lag_90", "national_return_7", "national_return_30",
    "state_national_spread", "state_national_ratio",
]
FEATURE_COLUMNS += [
    f"cross_{grain.lower()}_lag{lag}"
    for grain in TARGET_GRAINS
    for lag in CROSS_GRAIN_LAGS
]
FEATURE_COLUMNS += [f"spread_to_{grain.lower()}_lag1" for grain in TARGET_GRAINS]
FEATURE_COLUMNS += ["state_code"]

def mape(actual, predicted) -> float:
    actual = pd.to_numeric(pd.Series(actual), errors="coerce")
    predicted = pd.to_numeric(pd.Series(predicted), errors="coerce")
    mask = actual.abs() > 1e-9
    return math.inf if not mask.any() else float(((predicted[mask] - actual[mask]).abs() / actual[mask].abs()).mean() * 100)


def mae(actual, predicted) -> float:
    actual = pd.to_numeric(pd.Series(actual), errors="coerce")
    predicted = pd.to_numeric(pd.Series(predicted), errors="coerce")
    return float((predicted - actual).abs().mean())


def mase(actual, predicted, scale: float) -> float:
    return math.inf if not np.isfinite(scale) or scale <= 1e-9 else float(mae(actual, predicted) / scale)


def pinball_loss(actual, predicted, quantile: float) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = actual - predicted
    return float(np.nanmean(np.maximum(quantile * error, (quantile - 1.0) * error)))


def fill_features(frame: pd.DataFrame, fill_values: dict[str, float]) -> pd.DataFrame:
    out = frame.reindex(columns=FEATURE_COLUMNS).copy()
    out = out.replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0)
    out["state_code"] = pd.to_numeric(out["state_code"], errors="coerce").fillna(0).round().astype("int32")
    return out


def _rolling_slope(values: pd.Series, window: int) -> pd.Series:
    def slope(arr: np.ndarray) -> float:
        mask = np.isfinite(arr)
        if mask.sum() < 3:
            return np.nan
        x = np.arange(len(arr), dtype=float)[mask]
        y = arr[mask].astype(float)
        x = x - x.mean()
        y = y - y.mean()
        denom = float(np.dot(x, x))
        return np.nan if denom <= 1e-12 else float(np.dot(x, y) / denom)

    return values.rolling(window, min_periods=max(3, window // 3)).apply(slope, raw=True)


def _national_feature_table(df: pd.DataFrame) -> pd.DataFrame:
    dates = df[["date", "grain"]].drop_duplicates().sort_values(["grain", "date"])
    national = (
        df[df["state_name"].eq("All States")][["date", "grain", "price"]]
        .drop_duplicates(["date", "grain"], keep="last")
        .rename(columns={"price": "national_price"})
    )
    table = dates.merge(national, on=["date", "grain"], how="left").sort_values(["grain", "date"])
    table["national_price"] = table.groupby("grain", sort=False)["national_price"].ffill()
    group = table.groupby("grain", sort=False)
    for lag in [1, 7, 30, 90]:
        table[f"national_lag_{lag}"] = group["national_price"].shift(lag)
    table["national_return_7"] = group["national_price"].pct_change(7, fill_method=None).clip(-0.5, 0.5)
    table["national_return_30"] = group["national_price"].pct_change(30, fill_method=None).clip(-0.5, 0.5)
    return table


MSP_TABLE = {
    "Wheat": {
        2001: 580, 2002: 610, 2003: 620, 2004: 630, 2005: 640, 2006: 650,
        2007: 700, 2008: 750, 2009: 800, 2010: 850, 2011: 900, 2012: 965,
        2013: 1035, 2014: 1400, 2015: 1450, 2016: 1525, 2017: 1625, 2018: 1735,
        2019: 1840, 2020: 1925, 2021: 1975, 2022: 2015, 2023: 2125, 2024: 2275,
        2025: 2425, 2026: 2600,
    },
    "Paddy": {
        2001: 530, 2002: 530, 2003: 550, 2004: 560, 2005: 570, 2006: 580,
        2007: 645, 2008: 745, 2009: 950, 2010: 1000, 2011: 1080, 2012: 1250,
        2013: 1310, 2014: 1360, 2015: 1410, 2016: 1470, 2017: 1550, 2018: 1750,
        2019: 1815, 2020: 1868, 2021: 1940, 2022: 2040, 2023: 2183, 2024: 2300,
        2025: 2425, 2026: 2550,
    },
    "Maize": {
        2001: 330, 2002: 330, 2003: 330, 2004: 340, 2005: 350, 2006: 360,
        2007: 395, 2008: 415, 2009: 840, 2010: 880, 2011: 980, 2012: 1175,
        2013: 1310, 2014: 1310, 2015: 1325, 2016: 1365, 2017: 1425, 2018: 1700,
        2019: 1760, 2020: 1850, 2021: 1870, 2022: 1962, 2023: 2090, 2024: 2225,
        2025: 2350, 2026: 2465,
    },
    "Mustard": {
        2001: 1600, 2002: 1600, 2003: 1500, 2004: 1500, 2005: 1700, 2006: 1715,
        2007: 1800, 2008: 1830, 2009: 1830, 2010: 1830, 2011: 1850, 2012: 2500,
        2013: 3000, 2014: 3000, 2015: 3350, 2016: 3350, 2017: 3700, 2018: 4000,
        2019: 4200, 2020: 4425, 2021: 4650, 2022: 5050, 2023: 5450, 2024: 5650,
        2025: 5950, 2026: 6200,
    },
}

HARVEST_MONTHS = {
    "Wheat": [3, 4], "Paddy": [9, 10, 11], "Maize": [9, 10], "Mustard": [2, 3],
}
SOWING_MONTHS = {
    "Wheat": [10, 11, 12], "Paddy": [6, 7], "Maize": [6, 7], "Mustard": [10, 11],
}


def _months_to_next(month: int, months: list[int]) -> int:
    return min((int(target) - int(month)) % 12 for target in months) if months else 0



def _daily_fill_all_states(df: pd.DataFrame) -> pd.DataFrame:
    national_parts = []
    non_national = df[~df["state_name"].eq("All States")].copy()
    for grain, g in df[df["state_name"].eq("All States")].groupby("grain", sort=False):
        g = g.sort_values("date").drop_duplicates("date", keep="last").copy()
        if g.empty:
            continue
        full_index = pd.date_range(g["date"].min(), g["date"].max(), freq="D")
        g = g.set_index("date").reindex(full_index)
        g.index.name = "date"
        g["grain"] = grain
        g["state_name"] = "All States"
        g["state_id"] = g.get("state_id", pd.Series(index=g.index, dtype=object)).ffill().bfill().fillna("100006")
        g["state_key"] = "all-states"
        for column in ["price", "price_low", "price_high"]:
            if column in g:
                g[column] = pd.to_numeric(g[column], errors="coerce").ffill(limit=NATIONAL_FFILL_LIMIT_DAYS)
        for column in ["arrival", "market_count", "variety_count", "grade_count"]:
            if column in g:
                g[column] = pd.to_numeric(g[column], errors="coerce").fillna(0)
        g["is_observed"] = g.get("is_observed", pd.Series(index=g.index, dtype=object)).fillna(False).astype(bool)
        g["source"] = g.get("source", pd.Series(index=g.index, dtype=object)).ffill().bfill().fillna("national_daily_fill")
        g["source_priority"] = pd.to_numeric(g.get("source_priority", pd.Series(index=g.index)), errors="coerce").fillna(1).astype(int)
        if "source_fetched_at" in g:
            g["source_fetched_at"] = g["source_fetched_at"].ffill()
        national_parts.append(g.reset_index().dropna(subset=["price"]))
    if not national_parts:
        return df
    filled = pd.concat(national_parts, ignore_index=True, sort=False)
    return pd.concat([non_national, filled[df.columns]], ignore_index=True, sort=False)
def feature_engineering(canonical: pd.DataFrame) -> pd.DataFrame:
    df = canonical.copy()
    df["date"] = pd.to_datetime(df["date"])
    for column in ["price", "price_low", "price_high", "arrival", "market_count", "variety_count", "grade_count"]:
        df[column] = pd.to_numeric(df.get(column), errors="coerce")
    df["arrival"] = df["arrival"].fillna(0)
    df["market_count"] = df["market_count"].fillna(0)
    df["variety_count"] = df["variety_count"].fillna(0)
    df["grade_count"] = df["grade_count"].fillna(0)
    df = df.dropna(subset=["date", "state_name", "grain", "price"]).sort_values(["grain", "state_name", "date"])
    if NATIONAL_DAILY_FILL:
        before = len(df[df["state_name"].eq("All States")])
        df = _daily_fill_all_states(df)
        after = len(df[df["state_name"].eq("All States")])
        print(f"National daily fill: All States rows {before:,} -> {after:,} using {NATIONAL_FFILL_LIMIT_DAYS}-day ffill limit")

    national_features = _national_feature_table(df)
    df = df.merge(national_features, on=["date", "grain"], how="left")

    national_wide = (
        df[df["state_name"].eq("All States")]
        .pivot_table(index="date", columns="grain", values="price", aggfunc="last")
        .sort_index()
        .ffill()
    )
    for grain_name in TARGET_GRAINS:
        col = f"cross_{grain_name.lower()}"
        if grain_name in national_wide.columns:
            df = df.merge(national_wide[grain_name].rename(col).reset_index(), on="date", how="left")
        else:
            df[col] = np.nan

    group = df.groupby(["grain", "state_name"], sort=False)
    group_keys = [df["grain"], df["state_name"]]

    log_price = np.log(df["price"].clip(lower=1e-9))
    log_price_group = log_price.groupby(group_keys, sort=False)
    for lag in PRICE_LAGS:
        df[f"price_lag_{lag}"] = group["price"].shift(lag)
    for lag in DASHBOARD_PRICE_LAGS:
        df[f"log_price_lag_{lag}"] = log_price_group.shift(lag)

    previous_price = group["price"].shift(1)
    raw_log_return = np.log(df["price"] / previous_price.replace(0, np.nan)).clip(-0.5, 0.5)
    for lag in LOG_RETURN_LAGS:
        df[f"log_ret_lag_{lag}"] = raw_log_return.groupby(group_keys, sort=False).shift(lag)
    for period in [1, 3, 7, 14, 30, 90]:
        df[f"return_{period}"] = group["price"].pct_change(periods=period, fill_method=None).clip(-0.5, 0.5)

    shifted_price = group["price"].shift(1)
    for window in ROLL_WINDOWS:
        rolling = shifted_price.groupby(group_keys, sort=False)
        df[f"rolling_mean_{window}"] = rolling.transform(lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).mean())
        df[f"rolling_median_{window}"] = rolling.transform(lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).median())
        df[f"rolling_min_{window}"] = rolling.transform(lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).min())
        df[f"rolling_max_{window}"] = rolling.transform(lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).max())
        df[f"rolling_std_{window}"] = rolling.transform(lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).std())
        df[f"rolling_range_{window}"] = df[f"rolling_max_{window}"] - df[f"rolling_min_{window}"]
        df[f"rolling_cv_{window}"] = df[f"rolling_std_{window}"] / df[f"rolling_mean_{window}"].replace(0, np.nan)
        df[f"price_pct_range_{window}"] = (
            (previous_price - df[f"rolling_min_{window}"]) / df[f"rolling_range_{window}"].replace(0, np.nan)
        )
        df[f"vol_log_ret_{window}"] = raw_log_return.groupby(group_keys, sort=False).shift(1).groupby(group_keys, sort=False).transform(
            lambda s, w=window: s.rolling(w, min_periods=max(3, w // 3)).std()
        )

    for span in EWMA_SPANS:
        df[f"ewm_mean_{span}"] = group["price"].transform(lambda s, sp=span: s.shift(1).ewm(span=sp, adjust=False, min_periods=3).mean())
        df[f"ewma_ratio_{span}"] = previous_price / df[f"ewm_mean_{span}"].replace(0, np.nan)

    for window in TREND_WINDOWS:
        df[f"trend_slope_{window}"] = shifted_price.groupby(group_keys, sort=False).transform(lambda s, w=window: _rolling_slope(s, w))

    df["momentum_7_30"] = df["ewm_mean_7"] - df["ewm_mean_30"]
    df["momentum_30_90"] = df["ewm_mean_30"] - df["ewm_mean_90"]
    df["price_vs_mean_30"] = df["price"] / df["rolling_mean_30"].replace(0, np.nan)
    df["price_vs_mean_90"] = df["price"] / df["rolling_mean_90"].replace(0, np.nan)
    df["price_range_pct"] = (df["price_high"] - df["price_low"]) / df["price"].replace(0, np.nan)
    df["price_spread_lag1"] = (df["price_high"] - df["price_low"]).groupby(group_keys, sort=False).shift(1)
    df["price_spread_roll7"] = df["price_spread_lag1"].groupby(group_keys, sort=False).transform(lambda s: s.rolling(7, min_periods=2).mean())

    log_arrival = np.log1p(df["arrival"].clip(lower=0))
    for lag in ARRIVAL_LAGS:
        df[f"arrival_lag_{lag}"] = group["arrival"].shift(lag)
        df[f"log_arr_lag_{lag}"] = log_arrival.groupby(group_keys, sort=False).shift(lag)
    for window in ARRIVAL_ROLLS:
        df[f"arrival_rolling_mean_{window}"] = group["arrival"].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=max(2, w // 3)).mean())
        df[f"arrival_rolling_std_{window}"] = group["arrival"].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=max(3, w // 3)).std())
    df["arrival_trend_7"] = group["arrival"].transform(lambda s: _rolling_slope(s.shift(1), 7))
    df["arrival_trend_30"] = group["arrival"].transform(lambda s: _rolling_slope(s.shift(1), 30))
    df["arrival_stale"] = (df["arrival"].isna() | df["arrival"].eq(0)).astype(int)

    df["market_count_lag_1"] = group["market_count"].shift(1)
    df["market_count_lag_7"] = group["market_count"].shift(7)
    df["market_count_rolling_mean_7"] = group["market_count"].transform(lambda s: s.shift(1).rolling(7, min_periods=2).mean())
    df["market_count_rolling_mean_30"] = group["market_count"].transform(lambda s: s.shift(1).rolling(30, min_periods=3).mean())

    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["year_index"] = df["date"].dt.year - df["date"].dt.year.min()
    df["season_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["season_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["is_monsoon"] = df["month"].isin([6, 7, 8, 9]).astype(int)

    rabi_months = {"Wheat": {3, 4, 5}, "Paddy": {3, 4, 5}, "Maize": {2, 3, 4}, "Mustard": {2, 3, 4}}
    kharif_months = {"Wheat": set(), "Paddy": {10, 11, 12}, "Maize": {9, 10, 11}, "Mustard": set()}
    df["is_harvest_rabi"] = [int(month in rabi_months.get(grain, set())) for grain, month in zip(df["grain"], df["month"])]
    df["is_harvest_kharif"] = [int(month in kharif_months.get(grain, set())) for grain, month in zip(df["grain"], df["month"])]
    df["is_harvest_season"] = [int(month in HARVEST_MONTHS.get(grain, [])) for grain, month in zip(df["grain"], df["month"])]
    df["is_sowing_season"] = [int(month in SOWING_MONTHS.get(grain, [])) for grain, month in zip(df["grain"], df["month"])]
    df["months_to_harvest"] = [
        _months_to_next(month, HARVEST_MONTHS.get(grain, []))
        for grain, month in zip(df["grain"], df["month"])
    ]
    for period in FOURIER_PERIODS:
        for k in range(1, FOURIER_K + 1):
            angle = 2 * np.pi * k * df["day_of_year"] / period
            df[f"sin_{period}d_k{k}"] = np.sin(angle)
            df[f"cos_{period}d_k{k}"] = np.cos(angle)

    df["msp"] = [MSP_TABLE.get(grain, {}).get(int(year), np.nan) for grain, year in zip(df["grain"], df["date"].dt.year)]
    df["msp_gap_ratio"] = df["price"] / df["msp"].replace(0, np.nan)
    df["price_above_msp"] = (df["price"] > df["msp"]).astype(int)
    df["msp_pct_diff"] = (df["price"] - df["msp"]) / df["msp"].replace(0, np.nan) * 100

    df["state_national_spread"] = df["price"] - df["national_price"]
    df["state_national_ratio"] = df["price"] / df["national_price"].replace(0, np.nan)
    for grain_name in TARGET_GRAINS:
        col = f"cross_{grain_name.lower()}"
        for lag in CROSS_GRAIN_LAGS:
            df[f"{col}_lag{lag}"] = group[col].shift(lag)
        df[f"spread_to_{grain_name.lower()}_lag1"] = previous_price / group[col].shift(1).replace(0, np.nan)

    state_map = {state: idx for idx, state in enumerate(sorted(df["state_name"].astype(str).unique()))}
    df["state_code"] = df["state_name"].map(state_map).astype("int32")
    return df.replace([np.inf, -np.inf], np.nan)

def horizon_clip(prices: np.ndarray, values: np.ndarray, horizon: int) -> np.ndarray:
    low_ratio, high_ratio = HORIZON_PRICE_CLIP_BOUNDS.get(int(horizon), (0.55, 1.75))
    return np.clip(values, prices * low_ratio, prices * high_ratio)


def make_candidate_models(horizon: int, train_rows: int) -> dict[str, object]:
    models: dict[str, object] = {}
    seed = 42 + int(horizon)
    if CatBoostRegressor is not None:
        models["catboost"] = CatBoostRegressor(
            iterations=2000, learning_rate=0.01, depth=7, l2_leaf_reg=5.0,
            loss_function="RMSE", random_seed=seed, verbose=False,
            early_stopping_rounds=200, allow_writing_files=False,
        )
    if LGBMRegressor is not None:
        models["lightgbm"] = LGBMRegressor(
            n_estimators=3000, learning_rate=0.01, num_leaves=127,
            min_child_samples=50, subsample=0.85, colsample_bytree=0.85,
            reg_alpha=0.5, reg_lambda=5.0, objective="huber",
            random_state=seed, n_jobs=-1, verbose=-1,
        )
    if XGBRegressor is not None:
        models["xgboost"] = XGBRegressor(
            n_estimators=2000, learning_rate=0.01, max_depth=7,
            subsample=0.85, colsample_bytree=0.85, reg_alpha=0.5,
            reg_lambda=5.0, objective="reg:pseudohubererror",
            random_state=seed, tree_method="hist", n_jobs=-1,
            verbosity=0, early_stopping_rounds=200,
        )
    models["ridge"] = Ridge(alpha=100.0, random_state=seed)
    if INCLUDE_SKLEARN_FALLBACK_MODELS:
        models["hist_gb"] = HistGradientBoostingRegressor(
            max_iter=650, learning_rate=0.025, l2_regularization=0.10,
            max_leaf_nodes=63, random_state=seed, loss="squared_error",
        )
        models["extra_trees"] = ExtraTreesRegressor(
            n_estimators=220, min_samples_leaf=3, max_features=0.80,
            random_state=seed, n_jobs=-1,
        )
    return models


def _fit_model(
    name: str,
    model: object,
    X: pd.DataFrame,
    y: pd.Series,
    X_valid: pd.DataFrame | None = None,
    y_valid: pd.Series | None = None,
) -> object:
    if name == "ridge":
        scaler = StandardScaler()
        fitted = model.fit(scaler.fit_transform(X), y)
        return fitted, scaler
    if name == "catboost":
        kwargs = {"cat_features": ["state_code"]} if "state_code" in X.columns else {}
        if X_valid is not None and y_valid is not None and len(X_valid) > 0:
            kwargs.update({"eval_set": (X_valid, y_valid), "use_best_model": True})
        model.fit(X, y, **kwargs)
    elif name == "lightgbm":
        kwargs = {"categorical_feature": ["state_code"]} if "state_code" in X.columns else {}
        if X_valid is not None and y_valid is not None and len(X_valid) > 0:
            try:
                import lightgbm as lgb
                kwargs.update({
                    "eval_set": [(X_valid, y_valid)],
                    "callbacks": [lgb.early_stopping(200, verbose=False), lgb.log_evaluation(-1)],
                })
            except Exception:
                kwargs.update({"eval_set": [(X_valid, y_valid)]})
        model.fit(X, y, **kwargs)
    elif name == "xgboost":
        if X_valid is not None and y_valid is not None and len(X_valid) > 0:
            model.fit(X, y, eval_set=[(X_valid, y_valid)], verbose=False)
        else:
            model.fit(X, y, verbose=False)
    else:
        model.fit(X, y)
    return model


def _predict_model(model: object, X: pd.DataFrame) -> np.ndarray:
    if isinstance(model, tuple) and len(model) == 2 and hasattr(model[0], "predict"):
        fitted, scaler = model
        return np.asarray(fitted.predict(scaler.transform(X)), dtype=float)
    return np.asarray(model.predict(X), dtype=float)

def _build_tuned_model(name: str, trial, horizon: int):
    seed = 42 + int(horizon)
    if name == "hist_gb":
        return HistGradientBoostingRegressor(
            max_iter=trial.suggest_int("max_iter", 300, 1000),
            learning_rate=trial.suggest_float("learning_rate", 0.006, 0.06, log=True),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 24, 96),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 80),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 2.0, log=True),
            random_state=seed,
        )
    if name == "lightgbm" and LGBMRegressor is not None:
        return LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 500, 5000, step=500),
            learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
            num_leaves=trial.suggest_int("num_leaves", 31, 255),
            min_child_samples=trial.suggest_int("min_child_samples", 20, 150),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.0, 5.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.0, 10.0),
            objective="huber", random_state=seed, n_jobs=-1, verbose=-1,
        )
    if name == "xgboost" and XGBRegressor is not None:
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 500, 3500, step=250),
            max_depth=trial.suggest_int("max_depth", 4, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 25.0, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.0, 5.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.0, 10.0),
            objective="reg:pseudohubererror", random_state=seed, tree_method="hist",
            n_jobs=-1, verbosity=0, early_stopping_rounds=200,
        )
    if name == "catboost" and CatBoostRegressor is not None:
        return CatBoostRegressor(
            iterations=trial.suggest_int("iterations", 500, 3000, step=250),
            depth=trial.suggest_int("depth", 5, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
            random_strength=trial.suggest_float("random_strength", 0.0, 2.0),
            loss_function="RMSE", random_seed=seed, verbose=False,
            early_stopping_rounds=200, allow_writing_files=False,
        )
    return None

def tune_candidate_models(train: pd.DataFrame, valid: pd.DataFrame, fill_values: dict[str, float], horizon: int, transparent: bool = False) -> dict[str, object]:
    if not ENABLE_OPTUNA_TUNING or len(train) < 1000 or valid.empty:
        return {}
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except Exception:
        return {}
    X_train = fill_features(train[FEATURE_COLUMNS], fill_values)
    y_train = train["target_log_return"]
    X_valid = fill_features(valid[FEATURE_COLUMNS], fill_values)
    y_valid = valid["target_log_return"]
    valid_actual = valid["target_price"].to_numpy(dtype=float)
    valid_price = valid["price"].to_numpy(dtype=float)
    tuned: dict[str, object] = {}
    for name in OPTUNA_MODELS:
        if name not in {"hist_gb", "lightgbm", "xgboost", "catboost"}:
            continue
        if transparent:
            print(f"    -> Optuna tuning {name}: {OPTUNA_TRIALS} trials / {OPTUNA_TIMEOUT_SECONDS}s cap", flush=True)

        def objective(trial):
            model = _build_tuned_model(name, trial, horizon)
            if model is None:
                raise RuntimeError(f"{name} is unavailable")
            fitted = _fit_model(name, model, X_train, y_train, X_valid, y_valid)
            pred = horizon_clip(valid_price, valid_price * np.exp(np.clip(_predict_model(fitted, X_valid), -10, 15)), horizon)
            return mape(valid_actual, pred)

        try:
            study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42 + horizon))
            study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT_SECONDS, show_progress_bar=False, catch=(Exception,))
            if study.best_trial is not None:
                tuned_model = _build_tuned_model(name, study.best_trial, horizon)
                tuned[name] = tuned_model
                if transparent:
                    print(f"      best {name} MAPE={study.best_value:.3f}% | {study.best_params}", flush=True)
        except Exception as exc:
            print(f"Optuna skipped ({name}, {horizon}d): {type(exc).__name__}: {str(exc)[:140]}")
    return tuned

def train_candidate_models(train: pd.DataFrame, valid: pd.DataFrame, horizon: int, fill_values: dict[str, float], transparent: bool = False) -> dict[str, object]:
    fit_train = train
    if MAX_TRAIN_ROWS_PER_MODEL > 0 and len(fit_train) > MAX_TRAIN_ROWS_PER_MODEL:
        fit_train = fit_train.sample(MAX_TRAIN_ROWS_PER_MODEL, random_state=42 + horizon).sort_values("date")
    models = make_candidate_models(horizon, len(fit_train))
    models.update(tune_candidate_models(fit_train, valid, fill_values, horizon, transparent=transparent))
    X_train = fill_features(fit_train[FEATURE_COLUMNS], fill_values)
    y_train = fit_train["target_log_return"]
    X_valid = fill_features(valid[FEATURE_COLUMNS], fill_values) if valid is not None and len(valid) else None
    y_valid = valid["target_log_return"] if valid is not None and len(valid) else None
    fitted = {}
    for name, model in models.items():
        started = time.perf_counter()
        if transparent:
            print(f"    -> fitting {name} on {len(fit_train):,} rows ...", flush=True)
        try:
            fitted[name] = _fit_model(name, model, X_train, y_train, X_valid, y_valid)
            if transparent:
                print(f"      fitted {name} in {time.perf_counter() - started:,.1f}s", flush=True)
        except Exception as error:
            print(f"Model skipped ({name}, {horizon}d): {type(error).__name__}: {str(error)[:140]}")
    if not fitted:
        raise RuntimeError("No candidate model could be trained")
    return fitted

def predict_candidate_prices(models: dict[str, object], frame: pd.DataFrame, fill_values: dict[str, float], horizon: int) -> dict[str, np.ndarray]:
    X = fill_features(frame[FEATURE_COLUMNS], fill_values)
    prices = frame["price"].to_numpy(dtype=float)
    predictions = {"baseline": prices.copy()}
    for name, model in models.items():
        try:
            raw = prices * np.exp(np.clip(_predict_model(model, X), -10, 15))
            predictions[name] = horizon_clip(prices, raw, horizon)
        except Exception as error:
            print(f"Prediction skipped ({name}): {type(error).__name__}: {str(error)[:140]}")
    return predictions


def weighted_ensemble(predictions: dict[str, np.ndarray], actual: np.ndarray) -> tuple[np.ndarray, dict[str, float]]:
    model_names = [name for name in predictions if name != "baseline"]
    if not model_names:
        return predictions["baseline"], {"baseline": 1.0}
    scores = {name: mape(actual, predictions[name]) for name in model_names}
    finite_scores = {name: score for name, score in scores.items() if np.isfinite(score)}
    if not finite_scores:
        return predictions["baseline"], {"baseline": 1.0}
    best_score = min(finite_scores.values())
    keep = [name for name, score in finite_scores.items() if score <= best_score * ENSEMBLE_PRUNE_RATIO]
    if not keep:
        keep = [min(finite_scores, key=finite_scores.get)]
    weights_raw = np.array([1.0 / max(finite_scores[name], 0.01) for name in keep], dtype=float)
    weights_raw /= weights_raw.sum()
    ensemble = np.average(np.vstack([predictions[name] for name in keep]), axis=0, weights=weights_raw)
    return ensemble, {name: round(float(weight), 6) for name, weight in zip(keep, weights_raw)}

def align_future_targets(data: pd.DataFrame, horizon: int) -> pd.DataFrame:
    tolerance = pd.Timedelta(days=TARGET_MATCH_TOLERANCE_DAYS)
    aligned = []
    for _, state_df in data.groupby("state_name", sort=False):
        left = state_df.copy().sort_values("date")
        left["target_date"] = left["date"] + pd.to_timedelta(horizon, unit="D")
        right = state_df[["date", "price"]].copy().sort_values("date").rename(
            columns={"date": "actual_target_date", "price": "target_price"}
        )
        merged = pd.merge_asof(
            left.sort_values("target_date"), right,
            left_on="target_date", right_on="actual_target_date",
            direction="nearest", tolerance=tolerance,
        )
        aligned.append(merged)
    out = pd.concat(aligned, ignore_index=True) if aligned else pd.DataFrame()
    if out.empty:
        return out
    out["target_match_error_days"] = (out["actual_target_date"] - out["target_date"]).dt.days
    out["horizon"] = int(horizon)
    out["target_log_return"] = np.log(out["target_price"] / out["price"])
    return out.dropna(subset=["target_price", "target_log_return"]).sort_values(["date", "state_name"]).reset_index(drop=True)


def _bootstrap_lower_bound(improvements: np.ndarray, seed: int) -> float:
    values = np.asarray(improvements, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 4:
        return -math.inf
    rng = np.random.default_rng(seed)
    block = max(2, int(round(math.sqrt(len(values)))))
    means = []
    for _ in range(max(50, BOOTSTRAP_ITERATIONS)):
        starts = rng.integers(0, len(values), size=math.ceil(len(values) / block))
        sample = np.concatenate([values[(start + np.arange(block)) % len(values)] for start in starts])[:len(values)]
        means.append(float(np.mean(sample)))
    return float(np.quantile(means, 1.0 - GATE_CONFIDENCE_LEVEL))


def _fold_diagnostics(dates: pd.Series, baseline_error: np.ndarray, candidate_error: np.ndarray) -> tuple[int, int, list[float]]:
    frame = pd.DataFrame({"date": pd.to_datetime(dates).to_numpy(), "improvement": baseline_error - candidate_error}).sort_values("date")
    if frame.empty:
        return 0, 0, []
    folds = max(1, min(VALIDATION_FOLDS, len(frame)))
    chunks = np.array_split(frame, folds)
    fold_means = [float(chunk["improvement"].mean()) for chunk in chunks if not chunk.empty]
    return sum(value > 0 for value in fold_means), len(fold_means), fold_means


def _conformal_radius(actual: np.ndarray, predicted: np.ndarray) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = (actual > 0) & (predicted > 0) & np.isfinite(actual) & np.isfinite(predicted)
    if mask.sum() < 5:
        return float(np.log(1.25))
    residual = np.abs(np.log(actual[mask] / predicted[mask]))
    try:
        return float(np.quantile(residual, 1.0 - CONFORMAL_ALPHA, method="higher"))
    except TypeError:
        return float(np.quantile(residual, 1.0 - CONFORMAL_ALPHA, interpolation="higher"))


def row_payload(row: pd.Series, predicted: float, method: str, lower: float | None = None, upper: float | None = None) -> dict:
    payload = {
        "origin_date": row["date"].date().isoformat(),
        "target_date": row["actual_target_date"].date().isoformat(),
        "requested_target_date": row["target_date"].date().isoformat(),
        "target_match_error_days": int(row.get("target_match_error_days", 0)),
        "state_name": row["state_name"], "grain": row["grain"], "horizon": int(row["horizon"]),
        "actual_price": float(row["target_price"]), "predicted_price": float(predicted), "method": method,
    }
    if lower is not None and upper is not None:
        payload["prediction_lower"] = float(lower)
        payload["prediction_upper"] = float(upper)
        payload["interval_covered"] = bool(lower <= float(row["target_price"]) <= upper)
    return payload


def _apply_fixed_ensemble(predictions: dict[str, np.ndarray], weights: dict[str, float]) -> None:
    members = [name for name in weights if name in predictions and name != "baseline"]
    if not members:
        predictions["ensemble"] = predictions["baseline"].copy()
        return
    values = np.vstack([predictions[name] for name in members])
    member_weights = np.asarray([float(weights[name]) for name in members], dtype=float)
    if not np.isfinite(member_weights).all() or member_weights.sum() <= 0:
        predictions["ensemble"] = predictions["baseline"].copy()
        return
    predictions["ensemble"] = np.average(values, axis=0, weights=member_weights)


def _bounded_bias_factor(actual: np.ndarray, predicted: np.ndarray) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = (actual > 0) & (predicted > 0) & np.isfinite(actual) & np.isfinite(predicted)
    if valid.sum() < 5:
        return 1.0
    factor = float(np.exp(np.median(np.log(actual[valid] / predicted[valid]))))
    max_bias = max(0.0, float(MAX_BIAS_CORRECTION_PCT)) / 100.0
    return float(np.clip(factor, 1.0 - max_bias, 1.0 + max_bias))


def _unfitted_copy(name: str, fitted: object) -> object:
    from sklearn.base import clone

    template = fitted[0] if name == "ridge" and isinstance(fitted, tuple) else fitted
    try:
        copied = clone(template)
    except Exception:
        params = template.get_params() if hasattr(template, "get_params") else {}
        copied = template.__class__(**params)
    if name == "xgboost" and hasattr(copied, "set_params"):
        updates = {"early_stopping_rounds": None}
        best_iteration = getattr(template, "best_iteration", None)
        if best_iteration is not None:
            updates["n_estimators"] = max(1, int(best_iteration) + 1)
        copied.set_params(**updates)
    return copied


def refit_selected_models(
    fitted_models: dict[str, object],
    data: pd.DataFrame,
    fill_values: dict[str, float],
    selected_methods: set[str],
    ensemble_weights: dict[str, float],
) -> dict[str, object]:
    if not REFIT_SELECTED_MODELS_ON_FULL_DATA:
        return fitted_models

    required = {name for name in selected_methods if name not in {"baseline", "ensemble"}}
    if "ensemble" in selected_methods:
        required.update(name for name in ensemble_weights if name != "baseline")
    if not required:
        return {}

    fit_data = data
    if MAX_TRAIN_ROWS_PER_MODEL > 0 and len(fit_data) > MAX_TRAIN_ROWS_PER_MODEL:
        fit_data = fit_data.sample(MAX_TRAIN_ROWS_PER_MODEL, random_state=2026).sort_values("date")
    X_full = fill_features(fit_data[FEATURE_COLUMNS], fill_values)
    y_full = fit_data["target_log_return"]
    refitted = {}
    for name in sorted(required):
        if name not in fitted_models:
            continue
        try:
            refitted[name] = _fit_model(name, _unfitted_copy(name, fitted_models[name]), X_full, y_full)
        except Exception as exc:
            print(f"Production refit skipped ({name}): {type(exc).__name__}: {str(exc)[:140]}")
            refitted[name] = fitted_models[name]
    return refitted


def train_one(features: pd.DataFrame, grain: str, horizon: int, transparent: bool = False) -> dict | None:
    raw = features[features["grain"].eq(grain)].copy()
    if TRAINING_SCOPE == "national":
        raw = raw[raw["state_name"].eq("All States")].copy()
    counts = raw.groupby("state_name")["date"].count()
    serving_eligible_states = set(counts[counts >= MIN_STATE_OBSERVED_DAYS].index)
    if transparent:
        print(f"\n[{grain} | {horizon}d] feature rows: {len(raw):,}; series: {counts.size}", flush=True)
        print(f"  {len(serving_eligible_states)} series pass the serving-history gate", flush=True)
    if raw.empty:
        return None

    data = align_future_targets(raw, horizon)
    if transparent:
        survival = len(data) / max(len(raw), 1) * 100
        offsets = data["target_match_error_days"].value_counts().sort_index().to_dict() if not data.empty else {}
        print(f"  tolerance-aligned supervised rows: {len(data):,} ({survival:.1f}% survival); offsets={offsets}", flush=True)
    if len(data) < MIN_VALIDATION_SAMPLES * 4:
        return None

    latest_origin_date = pd.Timestamp(data["date"].max()).normalize()
    holdout_cutoff = latest_origin_date - pd.Timedelta(days=max(90, EVALUATION_HOLDOUT_DAYS) - 1)
    if len(data[data["date"] >= holdout_cutoff]) < MIN_VALIDATION_SAMPLES:
        holdout_ratio = min(0.35, max(0.10, EVALUATION_HOLDOUT_RATIO))
        holdout_cutoff = pd.Timestamp(data["date"].quantile(1.0 - holdout_ratio)).normalize()
    development = data[data["date"] < holdout_cutoff].copy()
    calibration_cutoff = holdout_cutoff - pd.Timedelta(days=max(90, ENSEMBLE_CALIBRATION_DAYS))
    if len(development[development["date"] >= calibration_cutoff]) < MIN_VALIDATION_SAMPLES:
        calibration_ratio = min(0.45, max(0.15, ENSEMBLE_CALIBRATION_RATIO))
        calibration_cutoff = pd.Timestamp(development["date"].quantile(1.0 - calibration_ratio)).normalize()

    pre_embargo_train = data[data["date"] < calibration_cutoff]
    train = data[
        (data["date"] < calibration_cutoff)
        & (data["actual_target_date"] < calibration_cutoff)
    ].copy()
    pre_embargo_calibration = data[
        (data["date"] >= calibration_cutoff)
        & (data["date"] < holdout_cutoff)
    ]
    calibration = data[
        (data["date"] >= calibration_cutoff)
        & (data["date"] < holdout_cutoff)
        & (data["actual_target_date"] < holdout_cutoff)
    ].copy()
    holdout = data[data["date"] >= holdout_cutoff].copy()
    embargo_removed = (len(pre_embargo_train) - len(train)) + (len(pre_embargo_calibration) - len(calibration))

    if transparent:
        print(
            f"  chronological split: train={len(train):,}, calibration={len(calibration):,}, "
            f"final holdout={len(holdout):,}, embargo removed={embargo_removed:,}",
            flush=True,
        )
        print(
            f"  train ends {train['date'].max().date()} | calibration starts {calibration['date'].min().date()} "
            f"| final holdout starts {holdout['date'].min().date()}",
            flush=True,
        )

    if (
        len(train) < MIN_VALIDATION_SAMPLES * 3
        or len(calibration) < MIN_VALIDATION_SAMPLES
        or len(holdout) < MIN_VALIDATION_SAMPLES
    ):
        return None

    evaluation_fill_values = (
        train[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .median(numeric_only=True)
        .fillna(0)
        .to_dict()
    )
    fitted_models = train_candidate_models(
        train,
        calibration,
        horizon,
        evaluation_fill_values,
        transparent=transparent,
    )

    calibration_predictions = predict_candidate_prices(
        fitted_models, calibration, evaluation_fill_values, horizon
    )
    ensemble_pred, ensemble_weights = weighted_ensemble(
        calibration_predictions,
        calibration["target_price"].to_numpy(dtype=float),
    )
    calibration_predictions["ensemble"] = ensemble_pred

    holdout_predictions = predict_candidate_prices(
        fitted_models, holdout, evaluation_fill_values, horizon
    )
    _apply_fixed_ensemble(holdout_predictions, ensemble_weights)

    gates: dict[str, dict] = {}
    calibration_series = {
        name: pd.Series(prediction, index=calibration.index)
        for name, prediction in calibration_predictions.items()
    }

    for state, state_calibration in calibration.groupby("state_name", sort=False):
        idx = state_calibration.index
        actual = state_calibration["target_price"].to_numpy(dtype=float)
        current = state_calibration["price"].to_numpy(dtype=float)
        method_bias_factors = {}
        calibrated_state_predictions = {}
        for name, series in calibration_series.items():
            raw_prediction = series.loc[idx].to_numpy(dtype=float)
            factor = 1.0 if name == "baseline" else _bounded_bias_factor(actual, raw_prediction)
            method_bias_factors[name] = factor
            calibrated_state_predictions[name] = horizon_clip(
                current,
                raw_prediction * factor,
                horizon,
            )
        method_scores = {
            name: mape(actual, prediction)
            for name, prediction in calibrated_state_predictions.items()
        }
        method_maes = {
            name: mae(actual, prediction)
            for name, prediction in calibrated_state_predictions.items()
        }
        baseline_pred = calibrated_state_predictions["baseline"]
        baseline_error = np.abs(baseline_pred - actual) / np.maximum(np.abs(actual), 1e-9) * 100
        robust_gate = {}
        passing = []

        for method, score in method_scores.items():
            if method == "baseline" or not np.isfinite(score):
                continue
            candidate_pred = calibrated_state_predictions[method]
            candidate_error = np.abs(candidate_pred - actual) / np.maximum(np.abs(actual), 1e-9) * 100
            improvement = baseline_error - candidate_error
            lower_bound = _bootstrap_lower_bound(
                improvement,
                seed=abs(hash((grain, horizon, state, method))) % (2**32),
            )
            fold_wins, fold_count, fold_means = _fold_diagnostics(
                state_calibration["actual_target_date"], baseline_error, candidate_error
            )
            required_wins = max(1, math.ceil(fold_count * MIN_TEMPORAL_FOLD_WIN_RATIO))
            absolute_gain = method_scores["baseline"] - score
            relative_gain = absolute_gain / max(method_scores["baseline"], 1e-9)
            passed = (
                lower_bound > MIN_MAPE_IMPROVEMENT
                and relative_gain >= MIN_RELATIVE_MAPE_IMPROVEMENT
                and fold_wins >= required_wins
            )
            robust_gate[method] = {
                "bootstrap_lower_improvement_pp": round(float(lower_bound), 4) if np.isfinite(lower_bound) else None,
                "absolute_mape_gain_pp": round(float(absolute_gain), 4),
                "relative_mape_gain": round(float(relative_gain), 6),
                "fold_wins": int(fold_wins),
                "fold_count": int(fold_count),
                "required_fold_wins": int(required_wins),
                "fold_mean_improvements_pp": [round(float(value), 4) for value in fold_means],
                "passed": bool(passed),
            }
            if passed:
                passing.append(method)

        selected = "baseline"
        reason = "no_calibration_candidate_passed"
        if state not in serving_eligible_states:
            reason = "thin_series_history"
        elif len(state_calibration) < MIN_VALIDATION_SAMPLES:
            reason = "insufficient_calibration"
        elif passing:
            selected = min(passing, key=lambda method: method_scores[method])
            if (
                "ensemble" in passing
                and method_scores["ensemble"] <= method_scores[selected] * DASHBOARD_ENSEMBLE_PREFERENCE_MARGIN
            ):
                selected = "ensemble"
            reason = "temporal_calibration_gate_passed"

        selected_calibration = calibrated_state_predictions[selected]
        radius = _conformal_radius(actual, selected_calibration)
        gates[state] = {
            "selected_method": selected,
            "candidate_method": min(
                (method for method in method_scores if method != "baseline"),
                key=lambda method: method_scores[method],
                default="baseline",
            ),
            "reason": reason,
            "selection_sample_count": int(len(state_calibration)),
            "selection_method_mapes": {
                name: round(float(score), 4)
                for name, score in method_scores.items()
                if np.isfinite(score)
            },
            "selection_method_maes": {
                name: round(float(score), 4)
                for name, score in method_maes.items()
                if np.isfinite(score)
            },
            "method_bias_factors": {
                name: round(float(factor), 8)
                for name, factor in method_bias_factors.items()
            },
            "price_bias_factor": round(float(method_bias_factors.get(selected, 1.0)), 8),
            "bias_calibration_method": "bounded_median_log_residual",
            "ensemble_weights": ensemble_weights,
            "robust_gate": robust_gate,
            "validation_strategy": "horizon_embargo_temporal_holdout",
            "training_end_date": train["date"].max().date().isoformat(),
            "calibration_start_date": calibration["date"].min().date().isoformat(),
            "calibration_end_date": calibration["date"].max().date().isoformat(),
            "validation_start_date": holdout["date"].min().date().isoformat(),
            "target_embargo_days": int(horizon),
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_log_radius": round(float(radius), 8),
            "interval_sample_count": int(len(state_calibration)),
            "interval_coverage_target": round(1.0 - CONFORMAL_ALPHA, 2),
        }

    validation_rows: list[dict] = []
    holdout_series = {
        name: pd.Series(prediction, index=holdout.index)
        for name, prediction in holdout_predictions.items()
    }
    holdout_evaluated_series = {
        name: series.copy()
        for name, series in holdout_series.items()
    }
    for state, state_holdout in holdout.groupby("state_name", sort=False):
        idx = state_holdout.index
        actual = state_holdout["target_price"].to_numpy(dtype=float)
        current = state_holdout["price"].to_numpy(dtype=float)
        gate = gates.setdefault(state, {
            "selected_method": "baseline",
            "reason": "missing_calibration_series",
            "selection_sample_count": 0,
            "selection_method_mapes": {},
            "selection_method_maes": {},
            "ensemble_weights": ensemble_weights,
            "robust_gate": {},
            "validation_strategy": "horizon_embargo_temporal_holdout",
            "conformal_log_radius": float(np.log(1.25)),
            "interval_sample_count": 0,
            "interval_coverage_target": round(1.0 - CONFORMAL_ALPHA, 2),
        })
        selected = gate.get("selected_method", "baseline")
        if selected not in holdout_series:
            selected = "baseline"
            gate["selected_method"] = selected
            gate["reason"] = "selected_model_unavailable_on_holdout"

        method_bias_factors = gate.get("method_bias_factors") or {}
        for name, series in holdout_series.items():
            factor = 1.0 if name == "baseline" else float(method_bias_factors.get(name, 1.0))
            adjusted = horizon_clip(current, series.loc[idx].to_numpy(dtype=float) * factor, horizon)
            holdout_evaluated_series[name].loc[idx] = adjusted
        holdout_mapes = {
            name: mape(actual, series.loc[idx].to_numpy(dtype=float))
            for name, series in holdout_evaluated_series.items()
        }
        holdout_maes = {
            name: mae(actual, series.loc[idx].to_numpy(dtype=float))
            for name, series in holdout_evaluated_series.items()
        }
        selected_pred = holdout_evaluated_series[selected].loc[idx].to_numpy(dtype=float)
        radius = float(gate.get("conformal_log_radius") or np.log(1.25))
        lower = selected_pred * np.exp(-radius)
        upper = selected_pred * np.exp(radius)
        coverage = float(np.mean((actual >= lower) & (actual <= upper))) if len(actual) else math.nan
        selected_mape = holdout_mapes.get(selected, math.inf)

        gate.update({
            "sample_count": int(len(state_holdout)),
            "baseline_mape": round(float(holdout_mapes.get("baseline", math.inf)), 4),
            "baseline_mae": round(float(holdout_maes.get("baseline", math.inf)), 4),
            "ml_mape": round(float(selected_mape), 4) if np.isfinite(selected_mape) else None,
            "ml_mae": round(float(holdout_maes.get(selected, math.nan)), 4) if np.isfinite(holdout_maes.get(selected, math.nan)) else None,
            "method_mapes": {
                name: round(float(score), 4)
                for name, score in holdout_mapes.items()
                if np.isfinite(score)
            },
            "method_maes": {
                name: round(float(score), 4)
                for name, score in holdout_maes.items()
                if np.isfinite(score)
            },
            "holdout_interval_coverage": round(coverage, 4) if np.isfinite(coverage) else None,
            "empirical_interval_coverage": round(coverage, 4) if np.isfinite(coverage) else None,
            "accuracy_target_mape": ACCURACY_TARGET_MAPE,
            "within_accuracy_target": bool(np.isfinite(selected_mape) and selected_mape <= ACCURACY_TARGET_MAPE),
        })

        for position, (_, row) in enumerate(state_holdout.iterrows()):
            item = row_payload(
                row,
                float(selected_pred[position]),
                selected,
                float(lower[position]),
                float(upper[position]),
            )
            item["evaluation_scope"] = "final_untouched_temporal_holdout"
            validation_rows.append(item)

    actual_all = holdout["target_price"].to_numpy(dtype=float)
    scale = float(
        train.sort_values(["state_name", "actual_target_date"])
        .groupby("state_name")["target_price"]
        .diff()
        .abs()
        .mean()
    )
    evaluated_holdout_predictions = {
        name: series.loc[holdout.index].to_numpy(dtype=float)
        for name, series in holdout_evaluated_series.items()
    }
    global_method_mapes = {
        name: round(mape(actual_all, prediction), 4)
        for name, prediction in evaluated_holdout_predictions.items()
    }
    global_method_mae = {
        name: round(mae(actual_all, prediction), 4)
        for name, prediction in evaluated_holdout_predictions.items()
    }
    global_method_mase = {
        name: round(mase(actual_all, prediction, scale), 4)
        for name, prediction in evaluated_holdout_predictions.items()
    }

    if transparent:
        score_preview = pd.DataFrame({
            "method": list(global_method_mapes),
            "final_holdout_MAPE_pct": list(global_method_mapes.values()),
            "final_holdout_MAE": [global_method_mae[name] for name in global_method_mapes],
            "final_holdout_MASE": [global_method_mase[name] for name in global_method_mapes],
        }).sort_values(["final_holdout_MAPE_pct", "final_holdout_MAE"])
        try:
            from IPython.display import display
            display(score_preview)
        except Exception:
            print(score_preview.to_string(index=False))

    production_fill_values = (
        data[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .median(numeric_only=True)
        .fillna(0)
        .to_dict()
    )
    selected_methods = {gate.get("selected_method", "baseline") for gate in gates.values()}
    production_models = refit_selected_models(
        fitted_models,
        data,
        production_fill_values,
        selected_methods,
        ensemble_weights,
    )

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / f"ensemble_{grain.lower()}_{horizon}d.pkl"
    with model_path.open("wb") as handle:
        pickle.dump({
            "models": production_models,
            "feature_fill_values": production_fill_values,
            "ensemble_weights": ensemble_weights,
            "horizon": horizon,
            "validation_strategy": "horizon_embargo_temporal_holdout",
        }, handle)

    return {
        "grain": grain,
        "horizon": horizon,
        "models": production_models,
        "model_path": str(model_path.name),
        "feature_columns": FEATURE_COLUMNS,
        "feature_fill_values": production_fill_values,
        "ensemble_weights": ensemble_weights,
        "gates": gates,
        "history_rows": [],
        "validation_rows": validation_rows,
        "efficiency_rows": validation_rows,
        "latest_training_date": data["date"].max().date().isoformat(),
        "global_method_mapes": global_method_mapes,
        "global_method_mae": global_method_mae,
        "global_method_mase": global_method_mase,
        "evaluation_strategy": "horizon_embargo_temporal_holdout",
        "split_info": {
            "training_end_date": train["date"].max().date().isoformat(),
            "calibration_start_date": calibration["date"].min().date().isoformat(),
            "calibration_end_date": calibration["date"].max().date().isoformat(),
            "validation_start_date": holdout["date"].min().date().isoformat(),
            "train_rows": int(len(train)),
            "calibration_rows": int(len(calibration)),
            "validation_rows": int(len(holdout)),
            "serving_eligible_states": int(len(serving_eligible_states)),
            "training_states": int(counts.size),
            "supervised_rows": int(len(data)),
            "raw_feature_rows": int(len(raw)),
            "target_match_tolerance_days": TARGET_MATCH_TOLERANCE_DAYS,
            "evaluation_holdout_days": int(EVALUATION_HOLDOUT_DAYS),
            "ensemble_calibration_days": int(ENSEMBLE_CALIBRATION_DAYS),
            "max_bias_correction_pct": float(MAX_BIAS_CORRECTION_PCT),
            "embargo_removed_rows": int(embargo_removed),
            "validation_strategy": "horizon_embargo_temporal_holdout",
            "target_match_offsets": {
                str(key): int(value)
                for key, value in data["target_match_error_days"].value_counts().sort_index().items()
            },
        },
    }

def train_models(canonical: pd.DataFrame, features: pd.DataFrame | None = None, transparent: bool = TRANSPARENT_MODE) -> dict:
    if features is None:
        if transparent:
            print("Engineering features inside train_models ...", flush=True)
        features = feature_engineering(canonical)
    elif transparent:
        print(f"Using precomputed feature table with {len(features):,} rows and {features.shape[1]} columns.", flush=True)
    registry = {"features": features, "models": {}, "history_rows": [], "validation_rows": [], "efficiency_rows": []}
    total_slots = len(TARGET_GRAINS) * len(HORIZONS)
    slot_number = 0
    for grain in TARGET_GRAINS:
        registry["models"][grain] = {}
        for horizon in HORIZONS:
            slot_number += 1
            if transparent:
                print("\n" + "=" * 90)
                print(f"TRAINING SLOT {slot_number}/{total_slots}: {grain} - {horizon}-day horizon")
                print("=" * 90, flush=True)
            started = time.perf_counter()
            trained = train_one(features, grain, horizon, transparent=transparent)
            elapsed = time.perf_counter() - started
            if trained:
                registry["models"][grain][str(horizon)] = trained
                registry["validation_rows"].extend(trained["validation_rows"])
                registry["efficiency_rows"].extend(trained.get("efficiency_rows", trained["validation_rows"]))
                print(f"Completed {grain} {horizon}d with {len(trained['models'])} fitted models in {elapsed:,.1f}s", flush=True)
                if transparent and "training_slot_snapshot" in globals():
                    training_slot_snapshot(trained, tail=TRANSPARENCY_VALIDATION_PLOT_TAIL)
            else:
                print(f"Using baseline only for {grain} {horizon}d; slot finished in {elapsed:,.1f}s", flush=True)
    return registry






## Source: predict

In [ ]:
from __future__ import annotations

import json
from datetime import timedelta

import numpy as np
import pandas as pd


def as_json_float(value: object) -> float | None:
    return None if pd.isna(value) else float(round(float(value), 4))


DRIVER_GROUPS = {
    "recent_state_price_momentum": ["return_1", "return_7", "return_30", "momentum_7_30", "price_vs_mean_30"],
    "national_price_relationship": ["national_price", "national_return_7", "national_return_30", "state_national_spread", "state_national_ratio"],
    "seasonality_and_harvest": ["season_sin", "season_cos", "month_sin", "month_cos", "is_harvest_rabi", "is_harvest_kharif", "is_monsoon"],
    "arrivals_and_market_coverage": ["arrival", "arrival_lag_7", "arrival_rolling_mean_30", "market_count", "market_count_rolling_mean_30", "variety_count", "grade_count"],
    "long_term_price_history": ["price_lag_30", "price_lag_90", "price_lag_180", "price_lag_365", "rolling_mean_90", "rolling_mean_180"],
}


def _method_log_return(trained: dict, method: str, features: pd.DataFrame) -> float | None:
    models = trained.get("models") or {}
    if method == "baseline":
        return 0.0
    if method == "ensemble":
        values, weights = [], []
        for name, weight in (trained.get("ensemble_weights") or {}).items():
            model = models.get(name)
            if model is None:
                continue
            values.append(float(_predict_model(model, features)[0]))
            weights.append(float(weight))
        return float(np.average(values, weights=weights)) if values and sum(weights) > 0 else None
    model = models.get(method)
    return None if model is None else float(_predict_model(model, features)[0])


def local_driver_scores(trained: dict | None, method: str, row: pd.Series) -> list[dict]:
    if not trained or method == "baseline":
        return [{"feature": "persistence_baseline", "score": 1.0}]
    fill_values = trained.get("feature_fill_values", {})
    base = fill_features(row[trained["feature_columns"]].to_frame().T, fill_values)
    base_pred = _method_log_return(trained, method, base)
    if base_pred is None:
        return [{"feature": "model_prediction", "score": 1.0}]
    impacts = []
    for label, columns in DRIVER_GROUPS.items():
        perturbed = base.copy()
        for column in columns:
            if column in perturbed:
                perturbed[column] = fill_values.get(column, 0)
        changed = _method_log_return(trained, method, perturbed)
        impacts.append((label, abs(base_pred - changed) if changed is not None else 0.0))
    total = sum(value for _, value in impacts)
    if total <= 1e-12:
        return [{"feature": "model_prediction", "score": 1.0}]
    return [
        {"feature": label, "score": round(float(value / total), 4)}
        for label, value in sorted(impacts, key=lambda item: item[1], reverse=True)
    ]


def interpolate_series(last_date, current_price: float, horizon_points: dict[int, float], horizon_intervals: dict[int, tuple[float, float]]) -> list[dict]:
    points = {0: current_price, **horizon_points}
    lower_points = {0: current_price, **{h: v[0] for h, v in horizon_intervals.items()}}
    upper_points = {0: current_price, **{h: v[1] for h, v in horizon_intervals.items()}}
    xs = np.array(sorted(points.keys()), dtype=float)
    ys = np.array([points[int(x)] for x in xs], dtype=float)
    lows = np.array([lower_points[int(x)] for x in xs], dtype=float)
    highs = np.array([upper_points[int(x)] for x in xs], dtype=float)
    series = []
    for day in range(1, int(xs.max()) + 1):
        series.append({
            "date": (last_date + timedelta(days=day)).isoformat(),
            "price": round(float(np.interp(day, xs, ys)), 2),
            "prediction_lower": round(float(np.interp(day, xs, lows)), 2),
            "prediction_upper": round(float(np.interp(day, xs, highs)), 2),
            "is_anchor": day in horizon_points, "anchor_horizon": day if day in horizon_points else None,
        })
    return series


def predict_method_price(
    trained: dict | None,
    method: str,
    row: pd.Series,
    current_price: float,
    horizon: int,
) -> tuple[float, float | None]:
    if not trained or method == "baseline":
        return current_price, None
    feature_frame = row[trained["feature_columns"]].to_frame().T
    if feature_frame.isna().all(axis=None):
        return current_price, None
    features = fill_features(feature_frame, trained.get("feature_fill_values", {}))
    low_ratio, high_ratio = HORIZON_PRICE_CLIP_BOUNDS.get(int(horizon), (0.55, 1.75))
    state_name = str(row.get("state_name", ""))
    gate = (trained.get("gates") or {}).get(state_name, {})
    bias_factor = float(gate.get("price_bias_factor") or 1.0)
    max_bias = max(0.0, float(MAX_BIAS_CORRECTION_PCT)) / 100.0
    bias_factor = float(np.clip(bias_factor, 1.0 - max_bias, 1.0 + max_bias))

    def calibrated_price(value: float) -> float:
        return float(np.clip(value * bias_factor, current_price * low_ratio, current_price * high_ratio))

    def model_price(model_name: str) -> float | None:
        model = (trained.get("models") or {}).get(model_name)
        if model is None:
            return None
        value = current_price * float(np.exp(np.clip(_predict_model(model, features)[0], -10, 15)))
        return float(np.clip(value, current_price * low_ratio, current_price * high_ratio))

    if method == "ensemble":
        values, weights = [], []
        for model_name, weight in (trained.get("ensemble_weights") or {}).items():
            value = model_price(model_name)
            if value is not None:
                values.append(value)
                weights.append(float(weight))
        if values and sum(weights) > 0:
            price = calibrated_price(float(np.average(values, weights=weights)))
            return price, price
        method = next(iter(trained.get("models") or {}), "baseline")
    price = model_price(method)
    if price is None:
        return current_price, None
    price = calibrated_price(price)
    return price, price

def normalize_live_dashboard_grain(value: object) -> str | None:
    text = str(value or "").strip().lower()
    if "wheat" in text:
        return "Wheat"
    if "paddy" in text:
        return "Paddy"
    if "maize" in text or "corn" in text:
        return "Maize"
    if "mustard" in text or "rapeseed" in text or "rape seed" in text:
        return "Mustard"
    return None


LIVE_DASHBOARD_PRICE_CACHE = None


def _extract_live_dashboard_prices(records: list, metadata: dict) -> dict:
    prices = {}
    for record in records or []:
        grain = normalize_live_dashboard_grain(
            record.get("commodity")
            or (record.get("raw") or {}).get("cmdt_name")
        )
        if not grain or grain in prices:
            continue
        price_block = record.get("price") or {}
        as_on = price_block.get("as_on") if isinstance(price_block, dict) else {}
        raw = record.get("raw") or {}
        price = as_on.get("value") if isinstance(as_on, dict) else None
        if price is None:
            price = raw.get("as_on_price")
        try:
            price = float(price)
        except Exception:
            continue
        if not np.isfinite(price) or price <= 0:
            continue
        prices[grain] = {
            "price": round(price, 2),
            "reported_dates": metadata.get("reported_dates") or [],
            "fetched_at": metadata.get("fetched_at"),
            "source": metadata.get("source"),
            "cache_key": metadata.get("cache_key"),
        }
    return prices


def fetch_live_dashboard_prices_from_website_api() -> dict:
    """Read the same backend endpoint that powers the visible website table."""
    import requests
    from datetime import datetime, timedelta
    from zoneinfo import ZoneInfo

    url = os.environ.get(
        "GRAINOLOGY_MARKETWISE_API_URL",
        "https://grainology.onrender.com/api/agmarknet/marketwise-price-arrival",
    )
    today = datetime.now(ZoneInfo("Asia/Kolkata")).date()
    base_payload = {
        "dashboard": "marketwise_price_arrival",
        "date": today.isoformat(),
        "state": 100006,
        "district": [],
        "market": [100009],
        "group": [],
        "commodity": [1, 2, 4],
        "variety": 100021,
        "grades": [4],
        "limit": 150,
        "force": False,
        "format": "json",
    }

    def _parse_reported_date(value: object):
        try:
            return datetime.strptime(str(value), "%d-%m-%Y").date()
        except Exception:
            return None

    def _parse_fetched_at(value: object):
        try:
            return datetime.fromisoformat(str(value).replace("Z", "+00:00")).timestamp()
        except Exception:
            return None

    def request_prices(payload: dict, label: str) -> tuple[dict, dict]:
        print(f"Website API live dashboard request payload ({label}):", payload)
        try:
            response = requests.post(url, json=payload, timeout=45)
            response.raise_for_status()
            data = response.json()
        except Exception as exc:
            print(f"Website API live price fetch skipped for {label}: {exc}")
            return {}, {}

        records = data.get("records") or []
        metadata = {
            "reported_dates": data.get("reported_dates") or [],
            "fetched_at": data.get("fetched_at"),
            "source": f"grainology_backend:{data.get('source') or 'unknown'}",
            "cache_key": f"website_api_marketwise_price_arrival:{label}:date={payload.get('date')}",
            "request_date": payload.get("date"),
            "record_count": len(records),
        }
        print(
            f"Website API live dashboard row ({label}):",
            {
                "reported_dates": metadata["reported_dates"],
                "fetched_at": metadata["fetched_at"],
                "source": metadata["source"],
                "record_count": len(records),
            },
        )
        return _extract_live_dashboard_prices(records, metadata), metadata

    candidates = []
    for offset in range(0, 4):
        request_date = (today - timedelta(days=offset)).isoformat()
        dated_payload = {**base_payload, "date": request_date}
        prices, metadata = request_prices(dated_payload, f"default_faq_d{offset}")
        mustard_payload = {**dated_payload, "commodity": [12]}
        mustard_prices, mustard_meta = request_prices(mustard_payload, f"mustard_faq_d{offset}")
        if "Mustard" not in mustard_prices:
            mustard_prices, mustard_meta = request_prices({**mustard_payload, "grades": []}, f"mustard_all_grades_d{offset}")
        if "Mustard" in mustard_prices:
            prices["Mustard"] = mustard_prices["Mustard"]
        reported = (metadata or mustard_meta or {}).get("reported_dates") or []
        reported_max = max([d for d in (_parse_reported_date(x) for x in reported) if d is not None], default=None)
        candidates.append({
            "request_date": request_date,
            "reported_max": reported_max,
            "count": len(prices),
            "prices": prices,
            "sources": [
                source
                for source in [
                    (metadata or {}).get("source"),
                    (mustard_meta or {}).get("source") if "Mustard" in prices else None,
                ]
                if source
            ],
            "latest_fetch": max(
                [
                    dt
                    for dt in [
                        _parse_fetched_at((metadata or {}).get("fetched_at")),
                        _parse_fetched_at((mustard_meta or {}).get("fetched_at")),
                    ]
                    if dt is not None
                ],
                default=None,
            ),
        })

    usable = [c for c in candidates if c["prices"]]
    if not usable:
        print("Website API returned no usable live dashboard prices")
        return {}

    complete = [c for c in usable if all(grain in c["prices"] for grain in TARGET_GRAINS)]
    pool = complete or usable
    newest_reported = max([c["reported_max"] for c in pool if c["reported_max"] is not None], default=None)
    if newest_reported is not None:
        pool = [c for c in pool if c["reported_max"] == newest_reported]
    all_cache_pool = [
        c for c in pool
        if c["sources"] and all(str(source).endswith(":cache") for source in c["sources"])
    ]
    if all_cache_pool:
        pool = all_cache_pool
    pool = sorted(
        pool,
        key=lambda c: (
            -c["count"],
            c["latest_fetch"] if c["latest_fetch"] is not None else float("inf"),
            c["request_date"],
        ),
    )
    chosen = pool[0]
    prices = chosen["prices"]
    print(
        "Selected live dashboard request date:",
        chosen["request_date"],
        "reported date:",
        chosen["reported_max"],
        "price count:",
        chosen["count"],
    )

    if prices:
        print("Website API live dashboard prices:", {k: v["price"] for k, v in sorted(prices.items())})
    else:
        print("Website API returned no usable live dashboard prices")
    return prices


def fetch_live_dashboard_prices_from_supabase_cache() -> dict:
    """Read the Market Wise cache as a fallback when the website API is unavailable.

    The model trains from canonical/agmarknet_ai_actuals, but the website's visible
    current All States prices come from agmarknet_marketwise_cache. This function
    keeps release current_price, chart hinge, forecast change %, and reasoning in
    parity with that table without changing the historical training contract.
    """
    client = get_supabase_client()
    if not client:
        return {}
    def _coerce_json(value, fallback):
        if value is None:
            return fallback
        if isinstance(value, str):
            try:
                return json.loads(value)
            except Exception:
                return fallback
        return value

    try:
        response = (
            client.table("agmarknet_marketwise_cache")
            .select("cache_key, request_payload, records, reported_dates, fetched_at, expires_at")
            .like("cache_key", "marketwise_price_arrival|%state=100006%")
            .order("fetched_at", desc=True)
            .limit(20)
            .execute()
        )
        rows = response.data or []
    except Exception as exc:
        print(f"Live dashboard price override skipped: {exc}")
        return {}

    if not rows:
        try:
            response = (
                client.table("agmarknet_marketwise_cache")
                .select("cache_key, request_payload, records, reported_dates, fetched_at, expires_at")
                .order("fetched_at", desc=True)
                .limit(100)
                .execute()
            )
            rows = response.data or []
            print("Exact All States cache key not found; scanning recent marketwise cache rows")
        except Exception as exc:
            print(f"Live dashboard price fallback scan skipped: {exc}")
            return {}

    def _row_score(row: dict) -> tuple:
        cache_key = str(row.get("cache_key") or "")
        payload = _coerce_json(row.get("request_payload"), {}) or {}
        records = _coerce_json(row.get("records"), []) or []
        is_marketwise = "marketwise_price_arrival" in cache_key or payload.get("dashboard") == "marketwise_price_arrival"
        is_all_states = payload.get("state") == 100006 or "state=100006" in cache_key
        grain_count = 0
        seen = set()
        for record in records:
            grain = normalize_live_dashboard_grain(record.get("commodity") or (record.get("raw") or {}).get("cmdt_name"))
            if grain:
                seen.add(grain)
        grain_count = len(seen)
        return (1 if is_marketwise else 0, 1 if is_all_states else 0, grain_count, str(row.get("fetched_at") or ""))

    rows = sorted(rows, key=_row_score, reverse=True)
    selected_row = rows[0] if rows else None
    if not selected_row:
        print("No marketwise cache rows were available for live dashboard price overrides")
        return {}

    selected_records = _coerce_json(selected_row.get("records"), []) or []
    print(
        "Selected live dashboard cache row:",
        {
            "cache_key": selected_row.get("cache_key"),
            "reported_dates": selected_row.get("reported_dates"),
            "fetched_at": selected_row.get("fetched_at"),
            "record_count": len(selected_records),
        },
    )

    prices = _extract_live_dashboard_prices(
        selected_records,
        {
            "reported_dates": selected_row.get("reported_dates") or [],
            "fetched_at": selected_row.get("fetched_at"),
            "source": "supabase_agmarknet_marketwise_cache",
            "cache_key": selected_row.get("cache_key"),
        },
    )
    missing_grains = [grain for grain in TARGET_GRAINS if grain not in prices]
    if missing_grains:
        print("Selected live dashboard cache row did not contain:", missing_grains)
    if prices:
        print("Live dashboard price overrides:", {k: v["price"] for k, v in sorted(prices.items())})
    else:
        print("No live dashboard price overrides found in agmarknet_marketwise_cache")
    return prices


def fetch_live_dashboard_prices_from_supabase() -> dict:
    global LIVE_DASHBOARD_PRICE_CACHE
    if LIVE_DASHBOARD_PRICE_CACHE:
        print("Reusing approved live dashboard prices from notebook sanity check")
        return LIVE_DASHBOARD_PRICE_CACHE

    prices = fetch_live_dashboard_prices_from_website_api()
    if prices:
        LIVE_DASHBOARD_PRICE_CACHE = prices
        return prices
    print("Falling back to Supabase agmarknet_marketwise_cache for live dashboard prices")
    prices = fetch_live_dashboard_prices_from_supabase_cache()
    if prices:
        LIVE_DASHBOARD_PRICE_CACHE = prices
    return prices


def apply_live_dashboard_price_overrides(predictions: dict, forecast_series: dict, actuals: dict) -> None:
    live_prices = fetch_live_dashboard_prices_from_supabase()
    if not live_prices:
        if str(os.environ.get("LIVE_DASHBOARD_PRICE_STRICT", "true")).strip().lower() in {"1", "true", "yes", "y"}:
            raise RuntimeError("Live dashboard price strict check failed: no live prices were found")
        return

    strict_live_prices = str(os.environ.get("LIVE_DASHBOARD_PRICE_STRICT", "true")).strip().lower() in {"1", "true", "yes", "y"}

    for grain, live in live_prices.items():
        state = "All States"
        payload = predictions.get(grain, {}).get(state)
        if not payload:
            continue

        old_current = float(payload.get("current_price") or 0)
        new_current = float(live["price"])
        if old_current <= 0 or not np.isfinite(old_current) or not np.isfinite(new_current):
            continue

        ratio = new_current / old_current
        payload["current_price"] = round(new_current, 2)
        payload["live_current_price"] = round(new_current, 2)
        payload["live_current_price_source"] = live.get("source")
        payload["live_current_price_dates"] = live.get("reported_dates") or []
        payload["live_current_price_fetched_at"] = live.get("fetched_at")
        payload["current_price_override_ratio"] = round(float(ratio), 8)

        for horizon_key, h_payload in (payload.get("horizons") or {}).items():
            for key in ["predicted_price", "model_price"]:
                if h_payload.get(key) is not None:
                    h_payload[key] = round(float(h_payload[key]) * ratio, 2)
            interval = h_payload.get("prediction_interval") or {}
            if interval.get("lower") is not None:
                interval["lower"] = round(float(interval["lower"]) * ratio, 2)
            if interval.get("upper") is not None:
                interval["upper"] = round(float(interval["upper"]) * ratio, 2)
            h_payload["live_current_price_aligned"] = True

        for point in forecast_series.get(grain, {}).get(state, []):
            for key in ["price", "prediction_lower", "prediction_upper", "lowerBound", "upperBound"]:
                if point.get(key) is not None:
                    point[key] = round(float(point[key]) * ratio, 2)

        context = actuals.get(grain, {}).get(state, {}).get("context") or []
        if context:
            context[-1]["price"] = round(new_current, 2)
            context[-1]["live_price_override"] = True
            context[-1]["live_price_source"] = live.get("source")

        print(f"Aligned {grain} / {state}: current_price {old_current:.2f} -> {new_current:.2f} (ratio {ratio:.6f})")

    audit_rows = []
    audit_errors = []
    for grain in TARGET_GRAINS:
        live = live_prices.get(grain)
        payload = predictions.get(grain, {}).get("All States")
        live_price = None if not live else float(live.get("price"))
        final_price = None if not payload else float(payload.get("current_price") or np.nan)
        delta = None if live_price is None or final_price is None or not np.isfinite(final_price) else round(final_price - live_price, 4)
        audit_rows.append({
            "grain": grain,
            "live_price": live_price,
            "final_prediction_current_price": final_price,
            "delta": delta,
            "source": None if not live else live.get("source"),
            "cache_key": None if not live else live.get("cache_key"),
        })
        if strict_live_prices:
            if live_price is None:
                audit_errors.append(f"{grain}: missing live dashboard price")
            elif final_price is None or not np.isfinite(final_price):
                audit_errors.append(f"{grain}: missing final prediction current_price")
            elif abs(final_price - live_price) > 0.01:
                audit_errors.append(f"{grain}: final current_price {final_price:.2f} != live price {live_price:.2f}")

    print("Final live dashboard price audit")
    try:
        display(pd.DataFrame(audit_rows))
    except Exception:
        print(audit_rows)

    if audit_errors:
        raise RuntimeError("Live dashboard price strict check failed: " + "; ".join(audit_errors))

def generate_predictions(canonical: pd.DataFrame, registry: dict) -> tuple[dict, dict, dict, dict]:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    features = registry["features"].copy()
    features["date"] = pd.to_datetime(features["date"])
    canonical_latest = pd.to_datetime(canonical["date"]).max().date()
    predictions, forecast_series, actuals, metrics = {}, {}, {}, {}
    latest_rows = features.sort_values("date").groupby(["grain", "state_name"], as_index=False).tail(1)
    for grain in TARGET_GRAINS:
        predictions[grain], forecast_series[grain], actuals[grain], metrics[grain] = {}, {}, {}, {}
        for _, row in latest_rows[latest_rows["grain"].eq(grain)].iterrows():
            state = row["state_name"]
            last_date = row["date"].date()
            current_price = float(row["price"])
            days_since = int((canonical_latest - last_date).days)
            status = "fresh" if days_since <= MAX_STATE_ACTUAL_STALENESS_DAYS else "stale"
            horizon_points, horizon_intervals = {}, {}
            predictions[grain][state] = {
                "current_price": round(current_price, 2), "last_actual_date": last_date.isoformat(),
                "days_since_last_actual": days_since, "forecast_start_date": (last_date + timedelta(days=1)).isoformat(),
                "status": status, "horizons": {},
            }
            metrics[grain][state] = {}
            for horizon in HORIZONS:
                trained = registry["models"].get(grain, {}).get(str(horizon))
                gate = trained["gates"].get(state) if trained else None
                selected_method = gate.get("selected_method", "baseline") if gate else "baseline"
                predicted_price, ml_price = predict_method_price(trained, selected_method, row, current_price, horizon)
                if gate is None and NATIONAL_PARITY_FORECASTS and trained and "All States" in trained.get("gates", {}):
                    national_gate = dict(trained["gates"].get("All States", {}))
                    national_method = national_gate.get("selected_method", "ensemble")
                    national_rows = latest_rows[(latest_rows["grain"].eq(grain)) & (latest_rows["state_name"].eq("All States"))]
                    if not national_rows.empty:
                        national_row = national_rows.iloc[-1]
                        national_current = float(national_row["price"])
                        national_predicted, _ = predict_method_price(trained, national_method, national_row, national_current, horizon)
                        ratio = national_predicted / max(national_current, 1e-9)
                        predicted_price = float(current_price * ratio)
                        ml_price = predicted_price
                        gate = national_gate
                        selected_method = f"national_parity_{national_method}"
                if ml_price is None and selected_method != "baseline":
                    selected_method = "baseline"
                radius = float((gate or {}).get("conformal_log_radius", np.log(1.25)))
                lower = float(predicted_price * np.exp(-radius))
                upper = float(predicted_price * np.exp(radius))
                low_ratio, high_ratio = HORIZON_PRICE_CLIP_BOUNDS.get(int(horizon), (0.55, 1.75))
                lower = max(lower, current_price * low_ratio)
                upper = min(upper, current_price * high_ratio)
                if lower > predicted_price:
                    lower = predicted_price
                if upper < predicted_price:
                    upper = predicted_price
                horizon_points[horizon] = predicted_price
                horizon_intervals[horizon] = (lower, upper)
                metric_payload = gate or {"selected_method": selected_method, "sample_count": 0, "reason": "no_trained_model"}
                metrics[grain][state][str(horizon)] = metric_payload
                predictions[grain][state]["horizons"][str(horizon)] = {
                    "target_date": (last_date + timedelta(days=horizon)).isoformat(),
                    "predicted_price": round(float(predicted_price), 2), "selected_method": selected_method,
                    "selection_reason": metric_payload.get("reason"),
                    "prediction_interval": {
                        "lower": round(lower, 2), "upper": round(upper, 2),
                        "coverage_target": round(1.0 - CONFORMAL_ALPHA, 2), "method": "split_conformal_log_residual",
                    },
                    "metrics": {
                        "mape": as_json_float(metric_payload.get("ml_mape")),
                        "mae": as_json_float(metric_payload.get("ml_mae")),
                        "baseline_mape": as_json_float(metric_payload.get("baseline_mape")),
                        "baseline_mae": as_json_float(metric_payload.get("baseline_mae")),
                        "sample_count": int(metric_payload.get("sample_count", 0)),
                        "method_mapes": metric_payload.get("method_mapes", {}),
                        "method_maes": metric_payload.get("method_maes", {}),
                        "empirical_interval_coverage": as_json_float(metric_payload.get("empirical_interval_coverage")),
                    },
                    "key_drivers": local_driver_scores(trained, selected_method, row),
                    "model_price": round(float(ml_price), 2) if ml_price is not None else None,
                }
            forecast_series[grain][state] = interpolate_series(last_date, current_price, horizon_points, horizon_intervals)

        grain_actuals = canonical[canonical["grain"].eq(grain)].copy()
        grain_actuals["date"] = pd.to_datetime(grain_actuals["date"])
        cutoff = grain_actuals["date"].max() - pd.Timedelta(days=FORECAST_HISTORY_DAYS)
        grain_actuals = grain_actuals[grain_actuals["date"].ge(cutoff)]
        for state, state_df in grain_actuals.groupby("state_name"):
            state_df = state_df.sort_values("date")
            actuals[grain][state] = {"context": [
                {"date": date.date().isoformat(), "price": round(float(price), 2), "is_observed": bool(is_observed)}
                for date, price, is_observed in zip(state_df["date"], state_df["price"], state_df["is_observed"])
            ]}
    apply_live_dashboard_price_overrides(predictions, forecast_series, actuals)
    (RELEASE_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "forecast_series.json").write_text(json.dumps(forecast_series, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "actuals.json").write_text(json.dumps(actuals, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")
    return predictions, forecast_series, actuals, metrics




## Source: efficiency

In [ ]:
from __future__ import annotations

import json
import math

import numpy as np
import pandas as pd


def metrics_for(rows: pd.DataFrame) -> dict:
    if rows.empty:
        return {"sample_count": 0}
    error = rows["predicted_price"] - rows["actual_price"]
    pct = (error.abs() / rows["actual_price"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    naive_scale = rows.sort_values("target_date")["actual_price"].diff().abs().mean()
    payload = {
        "sample_count": int(len(rows)), "mae": round(float(error.abs().mean()), 2),
        "rmse": round(float(math.sqrt((error ** 2).mean())), 2), "mape": round(float(pct.mean() * 100), 2),
        "wape": round(float(error.abs().sum() / rows["actual_price"].abs().sum() * 100), 2),
        "mase": round(float(error.abs().mean() / naive_scale), 4) if np.isfinite(naive_scale) and naive_scale > 0 else None,
    }
    if {"prediction_lower", "prediction_upper"}.issubset(rows.columns):
        covered = (rows["actual_price"] >= rows["prediction_lower"]) & (rows["actual_price"] <= rows["prediction_upper"])
        payload.update({
            "interval_coverage": round(float(covered.mean()), 4),
            "mean_interval_width": round(float((rows["prediction_upper"] - rows["prediction_lower"]).mean()), 2),
            "pinball_p10": round(pinball_loss(rows["actual_price"], rows["prediction_lower"], 0.10), 4),
            "pinball_p90": round(pinball_loss(rows["actual_price"], rows["prediction_upper"], 0.90), 4),
        })
    return payload


def generate_efficiency_data(registry: dict) -> tuple[dict, dict]:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    raw_rows = registry.get("efficiency_rows") or registry.get("validation_rows", [])
    rows = pd.DataFrame(raw_rows)
    historical_efficiency = {grain: {} for grain in TARGET_GRAINS}
    backtest = {grain: {} for grain in TARGET_GRAINS}
    if rows.empty:
        (RELEASE_DIR / "historical_efficiency.json").write_text(json.dumps(historical_efficiency, indent=2), encoding="utf-8")
        (RELEASE_DIR / "backtest.json").write_text(json.dumps(backtest, indent=2), encoding="utf-8")
        return historical_efficiency, backtest
    rows["error_pct"] = (rows["predicted_price"] - rows["actual_price"]).abs() / rows["actual_price"].replace(0, np.nan) * 100
    rows = rows.sort_values(["grain", "state_name", "horizon", "target_date"]).drop_duplicates(
        ["grain", "state_name", "horizon", "origin_date", "target_date"], keep="last"
    )
    for grain in TARGET_GRAINS:
        grain_rows = rows[rows["grain"].eq(grain)]
        for state, state_rows in grain_rows.groupby("state_name"):
            historical_efficiency[grain][state], backtest[grain][state] = {}, {}
            for horizon in HORIZONS:
                h_rows = state_rows[state_rows["horizon"].eq(horizon)].copy()
                if EFFICIENCY_MAX_ROWS_PER_SERIES > 0:
                    h_rows = h_rows.tail(EFFICIENCY_MAX_ROWS_PER_SERIES)
                if h_rows.empty:
                    continue
                series = []
                for row in h_rows.itertuples():
                    item = {
                        "origin_date": row.origin_date, "date": row.target_date,
                        "actual_price": round(float(row.actual_price), 2),
                        "predicted_price": round(float(row.predicted_price), 2),
                        "error_pct": round(float(row.error_pct), 2), "method": row.method,
                    }
                    if hasattr(row, "prediction_lower"):
                        item["prediction_lower"] = round(float(row.prediction_lower), 2)
                        item["prediction_upper"] = round(float(row.prediction_upper), 2)
                    series.append(item)
                payload = {
                    "horizon_days": horizon, "evaluation_scope": "full_history_selected_model_replay",
                    "is_true_model_backtest": False, "metrics": metrics_for(h_rows), "series": series,
                }
                historical_efficiency[grain][state][str(horizon)] = payload
                backtest[grain][state][str(horizon)] = {
                    "backtestDate": h_rows["target_date"].max(), "evaluationScope": payload["evaluation_scope"],
                    "metrics": payload["metrics"], "comparisons": [
                        {
                            "date": item["date"], "actualPrice": item["actual_price"],
                            "predictedPrice": item["predicted_price"],
                            "difference": round(item["predicted_price"] - item["actual_price"], 2),
                            "errorPct": item["error_pct"],
                        } for item in series[-15:]
                    ],
                }
    (RELEASE_DIR / "historical_efficiency.json").write_text(json.dumps(historical_efficiency, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "backtest.json").write_text(json.dumps(backtest, indent=2, allow_nan=False), encoding="utf-8")
    return historical_efficiency, backtest


## Source: evaluation


In [ ]:
from __future__ import annotations

import json
import math



def _finite(value):
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None
    return numeric if math.isfinite(numeric) else None


def generate_evaluation_report(registry: dict) -> dict:
    rows = []
    for grain in TARGET_GRAINS:
        for horizon in HORIZONS:
            trained = registry.get("models", {}).get(grain, {}).get(str(horizon)) or {}
            for state, gate in (trained.get("gates") or {}).items():
                baseline_mape = _finite(gate.get("baseline_mape"))
                model_mape = _finite(gate.get("ml_mape"))
                relative_gain = _finite(gate.get("relative_mape_gain"))
                rows.append({
                    "grain": grain,
                    "state": state,
                    "horizon_days": int(horizon),
                    "selected_method": gate.get("selected_method", "baseline"),
                    "candidate_method": gate.get("candidate_method"),
                    "sample_count": int(gate.get("sample_count", 0)),
                    "baseline_mape": baseline_mape,
                    "model_mape": model_mape,
                    "relative_mape_improvement_pct": None if relative_gain is None else round(relative_gain * 100, 2),
                    "fold_wins": int(gate.get("fold_wins", 0)),
                    "fold_count": int(gate.get("fold_count", 0)),
                    "validation_strategy": gate.get("validation_strategy"),
                    "training_end_date": gate.get("training_end_date"),
                    "calibration_start_date": gate.get("calibration_start_date"),
                    "validation_start_date": gate.get("validation_start_date"),
                    "target_embargo_days": int(gate.get("target_embargo_days", horizon)),
                    "interval_sample_count": int(gate.get("interval_sample_count", 0)),
                    "interval_coverage_target": _finite(gate.get("interval_coverage_target")),
                    "holdout_interval_coverage": _finite(gate.get("holdout_interval_coverage")),
                })

    national = [row for row in rows if row["state"] == "All States" and row["model_mape"] is not None]
    weighted_samples = sum(max(0, row["sample_count"]) for row in national)
    weighted_mape = (
        sum(row["model_mape"] * max(0, row["sample_count"]) for row in national) / weighted_samples
        if weighted_samples else None
    )
    report = {
        "schema_version": "1.0",
        "evaluation_strategy": "horizon_embargo_temporal_holdout",
        "leakage_controls": [
            "lagged and rolling features use prior observations only",
            "feature imputation statistics are learned from training rows only",
            "training target dates end before calibration begins",
            "Optuna tuning uses an inner training window",
            "ensemble weights are learned on calibration rows, not final holdout rows",
            "prediction intervals are calibrated before the final holdout using split conformal log residuals",
            "method promotion requires improvement across temporal holdout folds",
        ],
        "summary": {
            "series_evaluated": len(rows),
            "national_series_evaluated": len(national),
            "national_weighted_mape": None if weighted_mape is None else round(weighted_mape, 4),
            "national_min_mape": None if not national else round(min(row["model_mape"] for row in national), 4),
            "national_max_mape": None if not national else round(max(row["model_mape"] for row in national), 4),
            "promoted_series": sum(1 for row in rows if row["selected_method"] != "baseline"),
            "baseline_series": sum(1 for row in rows if row["selected_method"] == "baseline"),
        },
        "series": rows,
    }
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    (RELEASE_DIR / "evaluation_report.json").write_text(
        json.dumps(report, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    return report


## Source: drift


In [ ]:
from __future__ import annotations

import json

import numpy as np
import pandas as pd



def _safe_float(value):
    if value is None or pd.isna(value) or not np.isfinite(float(value)):
        return None
    return round(float(value), 4)


def generate_data_drift_report(
    canonical: pd.DataFrame,
    recent_days: int = 30,
    reference_days: int = 180,
) -> dict:
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["price"] = pd.to_numeric(frame["price"], errors="coerce")
    frame["arrival"] = pd.to_numeric(frame.get("arrival"), errors="coerce")
    latest = frame["date"].max()
    recent_start = latest - pd.Timedelta(days=recent_days - 1)
    reference_start = recent_start - pd.Timedelta(days=reference_days)
    rows = []

    for (grain, state), series in frame.groupby(["grain", "state_name"], observed=True):
        recent = series[series["date"].ge(recent_start)]
        reference = series[series["date"].ge(reference_start) & series["date"].lt(recent_start)]
        recent_price = recent["price"].median()
        reference_price = reference["price"].median()
        price_shift = (
            (recent_price - reference_price) / reference_price
            if pd.notna(recent_price) and pd.notna(reference_price) and reference_price != 0
            else np.nan
        )
        recent_arrival = recent["arrival"].median()
        reference_arrival = reference["arrival"].median()
        arrival_shift = (
            (recent_arrival - reference_arrival) / abs(reference_arrival)
            if pd.notna(recent_arrival) and pd.notna(reference_arrival) and reference_arrival != 0
            else np.nan
        )
        latest_series_date = series["date"].max()
        stale_days = int((latest - latest_series_date).days) if pd.notna(latest_series_date) else None
        severity = "ok"
        reasons = []
        if len(recent) < max(3, recent_days // 5):
            severity = "warning"
            reasons.append("low_recent_coverage")
        if stale_days is not None and stale_days > 14:
            severity = "warning"
            reasons.append("stale_series")
        if pd.notna(price_shift) and abs(price_shift) > 0.35:
            severity = "warning"
            reasons.append("large_price_distribution_shift")
        rows.append({
            "grain": str(grain),
            "state": str(state),
            "latest_date": None if pd.isna(latest_series_date) else latest_series_date.date().isoformat(),
            "stale_days": stale_days,
            "recent_rows": int(len(recent)),
            "reference_rows": int(len(reference)),
            "recent_median_price": _safe_float(recent_price),
            "reference_median_price": _safe_float(reference_price),
            "price_median_shift_pct": None if pd.isna(price_shift) else round(float(price_shift) * 100, 2),
            "arrival_median_shift_pct": None if pd.isna(arrival_shift) else round(float(arrival_shift) * 100, 2),
            "severity": severity,
            "reasons": reasons,
        })

    warnings = [row for row in rows if row["severity"] != "ok"]
    report = {
        "schema_version": "1.0",
        "latest_date": latest.date().isoformat(),
        "recent_window_days": int(recent_days),
        "reference_window_days": int(reference_days),
        "summary": {
            "series_checked": len(rows),
            "warning_series": len(warnings),
            "warning_ratio": round(len(warnings) / len(rows), 4) if rows else 0,
        },
        "alerts": warnings,
        "series": rows,
    }
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    (RELEASE_DIR / "data_drift_report.json").write_text(
        json.dumps(report, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    return report


## Source: reasoning

In [ ]:
from __future__ import annotations

import json


def generate_reasoning(predictions: dict, metrics: dict) -> dict:
    reasoning = {grain: {} for grain in TARGET_GRAINS}
    for grain, states in predictions.items():
        for state, payload in states.items():
            current = payload.get("current_price") or 0
            freshness = payload.get("status", "unknown")
            days_since = payload.get("days_since_last_actual", 0)
            reasoning[grain][state] = {}
            for horizon in HORIZONS:
                horizon_key = str(horizon)
                h_payload = payload.get("horizons", {}).get(horizon_key)
                if not h_payload:
                    continue
                predicted = h_payload["predicted_price"]
                method = h_payload.get("selected_method", "baseline")
                change_pct = ((predicted - current) / current * 100) if current else 0
                direction = "rise" if change_pct >= 0 else "fall"
                state_metrics = metrics.get(grain, {}).get(state, {}).get(horizon_key, {})
                sample_count = state_metrics.get("sample_count", 0)
                baseline_mape = state_metrics.get("baseline_mape")
                ml_mape = state_metrics.get("ml_mape")
                interval = h_payload.get("prediction_interval") or {}
                drivers = h_payload.get("key_drivers") or [{"feature": "persistence_baseline", "score": 1.0}]
                text = (
                    f"For {state}, {grain} is projected to {direction} by {abs(change_pct):.1f}% over {horizon} days. "
                    f"The selected method is {method}, chosen by an embargoed, bootstrap-and-fold stability gate."
                )
                if sample_count:
                    text += f" The gate used {sample_count} holdout samples."
                if ml_mape is not None and baseline_mape is not None:
                    text += f" Selected-method MAPE was {ml_mape:.2f}% versus persistence baseline {baseline_mape:.2f}%."
                if interval.get("lower") is not None:
                    text += f" The calibrated {int((interval.get('coverage_target', 0.9))*100)}% interval is Rs {interval['lower']:.2f}- Rs {interval['upper']:.2f}."
                if freshness == "stale":
                    text += f" Caution: the latest actual for this state is {days_since} days behind the dataset maximum."
                reasoning[grain][state][horizon_key] = {
                    "source": "model_perturbation_and_validation_gate", "text": text,
                    "key_drivers": drivers, "freshness_status": freshness,
                }
    (RELEASE_DIR / "reasoning.json").write_text(json.dumps(reasoning, indent=2, allow_nan=False), encoding="utf-8")
    return reasoning


## Source: manifest

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import uuid
import zipfile
from datetime import datetime, timezone

import pandas as pd


def sha256(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_states(canonical: pd.DataFrame) -> list[str]:
    states = sorted(canonical["state_name"].dropna().unique().tolist(), key=lambda state: (state != "All States", state))
    payload = {"states": [
        {"state_name": state, "state_key": "all-states" if state == "All States" else state.lower().replace(" ", "-")}
        for state in states
    ]}
    (RELEASE_DIR / "states.json").write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    return states


def mirror_release_for_kaggle_output() -> None:
    files = [path for path in sorted(RELEASE_DIR.iterdir()) if path.is_file()]
    zip_path = WORK_ROOT / "grainology_release.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            shutil.copy(path, WORK_ROOT / path.name)
            archive.write(path, arcname=f"release/{path.name}")
    print(f"Mirrored {len(files)} release files to {WORK_ROOT}")
    print(f"Release archive written to {zip_path}")


def evaluate_release_quality(canonical: pd.DataFrame, generated_at: str) -> dict:
    critical, warnings = [], []
    predictions = json.loads((RELEASE_DIR / "predictions.json").read_text(encoding="utf-8"))
    metrics = json.loads((RELEASE_DIR / "metrics.json").read_text(encoding="utf-8"))
    generated_date = pd.Timestamp(generated_at).date()
    latest_date = pd.to_datetime(canonical["date"]).max().date()
    staleness_days = max(0, (generated_date - latest_date).days)
    if staleness_days > MAX_DATA_STALENESS_DAYS:
        critical.append(f"canonical data is {staleness_days} days stale (limit {MAX_DATA_STALENESS_DAYS})")

    total_slots = ml_slots = interval_slots = 0
    max_change_seen = 0.0
    for grain, states in predictions.items():
        for state, payload in states.items():
            current = float(payload.get("current_price") or 0)
            if not math.isfinite(current) or current <= 0:
                critical.append(f"invalid current price for {grain}/{state}")
                continue
            for horizon_key, hp in (payload.get("horizons") or {}).items():
                total_slots += 1
                horizon = int(horizon_key)
                predicted = float(hp.get("predicted_price") or 0)
                if not math.isfinite(predicted) or predicted <= 0:
                    critical.append(f"invalid prediction for {grain}/{state}/{horizon}d")
                    continue
                change_pct = abs(predicted / current - 1.0) * 100
                max_change_seen = max(max_change_seen, change_pct)
                if change_pct > MAX_FORECAST_CHANGE_PCT.get(horizon, 150.0):
                    critical.append(f"forecast swing {change_pct:.1f}% exceeds limit for {grain}/{state}/{horizon}d")
                method = hp.get("selected_method", "baseline")
                ml_slots += int(method != "baseline")
                interval = hp.get("prediction_interval") or {}
                if interval:
                    lower, upper = interval.get("lower"), interval.get("upper")
                    if lower is None or upper is None or not (float(lower) <= predicted <= float(upper)):
                        critical.append(f"invalid interval for {grain}/{state}/{horizon}d")
                    else:
                        interval_slots += 1
                gate = metrics.get(grain, {}).get(state, {}).get(horizon_key, {})
                bm, mm = gate.get("baseline_mape"), gate.get("ml_mape")
                if method != "baseline" and bm is not None and mm is not None and float(mm) >= float(bm):
                    critical.append(f"selected ML does not beat baseline for {grain}/{state}/{horizon}d")

    ml_share = ml_slots / max(total_slots, 1)
    if ml_share < MIN_ML_SLOT_SHARE:
        warnings.append(f"ML selected for only {ml_share:.1%} of forecast slots")
    efficiency_mb = (RELEASE_DIR / "historical_efficiency.json").stat().st_size / (1024 * 1024)
    if efficiency_mb > MAX_EFFICIENCY_FILE_MB:
        critical.append(f"historical_efficiency.json is {efficiency_mb:.1f}MB (limit {MAX_EFFICIENCY_FILE_MB:.1f}MB)")
    return {
        "passed": not critical, "critical_failures": critical, "warnings": warnings,
        "data_staleness_days": staleness_days, "forecast_slots": total_slots,
        "ml_selected_share": round(ml_share, 6), "interval_coverage_share": round(interval_slots / max(total_slots, 1), 6),
        "max_absolute_forecast_change_pct": round(max_change_seen, 3),
        "historical_efficiency_size_mb": round(efficiency_mb, 3),
    }


def finalize_release(canonical: pd.DataFrame) -> dict:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(CANONICAL_PARQUET, RELEASE_DIR / "canonical_daily.parquet")
    shutil.copy(CANONICAL_CSV, RELEASE_DIR / "canonical_daily.csv")
    states = write_states(canonical)
    missing_before_manifest = [name for name in REQUIRED_RELEASE_FILES if name not in {"manifest.json", "checksums.json"} and not (RELEASE_DIR / name).exists()]
    if missing_before_manifest:
        raise ValueError(f"Release bundle missing required files before manifest: {missing_before_manifest}")

    run_id = os.environ.get("KAGGLE_KERNEL_RUN_ID")
    try:
        run_id = str(uuid.UUID(str(run_id)))
    except Exception:
        run_id = str(uuid.uuid4())
    generated_at = datetime.now(timezone.utc).isoformat()
    data_latest_date = pd.to_datetime(canonical["date"]).max().date().isoformat()
    quality = evaluate_release_quality(canonical, generated_at)

    files = {
        path.name: sha256(path) for path in RELEASE_DIR.iterdir()
        if path.is_file() and path.name not in {"manifest.json", "checksums.json"}
    }
    checksums = dict(sorted(files.items()))
    (RELEASE_DIR / "checksums.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")
    files["checksums.json"] = sha256(RELEASE_DIR / "checksums.json")
    manifest = {
        "schema_version": RELEASE_SCHEMA_VERSION, "canonical_schema_version": CANONICAL_SCHEMA_VERSION,
        "run_id": run_id, "status": "success" if quality["passed"] else "failed_quality_gate",
        "generated_at": generated_at, "data_latest_date": data_latest_date,
        "actuals_max_updated_at": generated_at, "actuals_row_count": int(len(canonical)),
        "forecast_start_date": (pd.to_datetime(data_latest_date) + pd.Timedelta(days=1)).date().isoformat(),
        "model_mode": MODEL_MODE, "grains": TARGET_GRAINS, "horizons": HORIZONS, "states": states,
        "aggregation_method": os.environ.get("AGGREGATION_METHOD", "median"),
        "code_version": os.environ.get("GITHUB_SHA") or os.environ.get("KAGGLE_KERNEL_RUN_ID") or "kaggle-local",
        "quality_gates": quality, "evaluation_strategy": "horizon_embargo_temporal_holdout", "market_model_mode": os.environ.get("MARKET_MODEL_MODE", "not_generated"), "market_level_available": (RELEASE_DIR / "market_predictions.json").exists(), "files": dict(sorted(files.items())),
    }
    (RELEASE_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding="utf-8")
    missing = [name for name in REQUIRED_RELEASE_FILES if not (RELEASE_DIR / name).exists()]
    if missing:
        raise ValueError(f"Release bundle missing required files: {missing}")
    if not quality["passed"] and FAIL_ON_QUALITY_GATE:
        raise ValueError("Release quality gate failed: " + "; ".join(quality["critical_failures"][:12]))
    mirror_release_for_kaggle_output()
    print(f"Release finalized at {RELEASE_DIR}")
    print(f"Quality gates: {'PASSED' if quality['passed'] else 'WARN/FAILED'}")
    return manifest


## Source: diagnostics

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None



def display_frame(frame: pd.DataFrame, rows: int = 20):
    try:
        from IPython.display import display
        display(frame.head(rows))
    except Exception:
        print(frame.head(rows).to_string(index=False))


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def canonical_summary(canonical: pd.DataFrame) -> pd.DataFrame:
    print_section("Canonical Dataset Summary")
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    summary = pd.DataFrame([{
        "rows": len(frame),
        "min_date": frame["date"].min().date().isoformat(),
        "max_date": frame["date"].max().date().isoformat(),
        "grains": frame["grain"].nunique(),
        "states": frame["state_name"].nunique(),
        "observed_rows": int(frame["is_observed"].fillna(False).sum()) if "is_observed" in frame else None,
    }])
    display_frame(summary)

    by_grain = (
        frame.groupby("grain")
        .agg(rows=("price", "size"), min_date=("date", "min"), max_date=("date", "max"), states=("state_name", "nunique"), avg_price=("price", "mean"))
        .reset_index()
    )
    by_grain["min_date"] = by_grain["min_date"].dt.date.astype(str)
    by_grain["max_date"] = by_grain["max_date"].dt.date.astype(str)
    by_grain["avg_price"] = by_grain["avg_price"].round(2)
    print("\nRows by grain")
    display_frame(by_grain, rows=50)

    latest_date = frame["date"].max()
    latest_rows = frame[frame["date"].eq(latest_date)].groupby("grain").agg(rows=("price", "size"), states=("state_name", "nunique")).reset_index()
    print(f"\nLatest canonical date: {latest_date.date().isoformat()}")
    display_frame(latest_rows, rows=50)
    return by_grain


def recent_data_summary(canonical: pd.DataFrame, days: int = 14) -> pd.DataFrame:
    print_section(f"Recently Added / Latest {days} Days")
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    cutoff = frame["date"].max() - pd.Timedelta(days=days - 1)
    recent = frame[frame["date"].ge(cutoff)]
    summary = (
        recent.groupby(["date", "grain"])
        .agg(rows=("price", "size"), states=("state_name", "nunique"), avg_price=("price", "mean"), observed=("is_observed", "sum"))
        .reset_index()
        .sort_values(["date", "grain"], ascending=[False, True])
    )
    summary["date"] = summary["date"].dt.date.astype(str)
    summary["avg_price"] = summary["avg_price"].round(2)
    display_frame(summary, rows=days * len(TARGET_GRAINS))
    return summary


def missingness_summary(canonical: pd.DataFrame) -> pd.DataFrame:
    print_section("Missingness / Data Quality")
    cols = ["price", "price_low", "price_high", "arrival", "market_count", "source", "is_observed"]
    available = [col for col in cols if col in canonical.columns]
    summary = pd.DataFrame({
        "column": available,
        "missing_pct": [round(float(canonical[col].isna().mean() * 100), 2) for col in available],
        "non_null": [int(canonical[col].notna().sum()) for col in available],
    })
    display_frame(summary, rows=50)
    duplicate_count = int(canonical.duplicated(["date", "state_name", "grain"]).sum())
    print(f"Duplicate date/state/grain rows: {duplicate_count}")
    return summary


def plot_history(canonical: pd.DataFrame, states: list[str] | None = None, grains: list[str] | None = None) -> None:
    print_section("Historical Price Trend")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    states = states or ["All States"]
    grains = grains or TARGET_GRAINS
    plot_df = frame[frame["state_name"].isin(states) & frame["grain"].isin(grains)].sort_values("date")
    if plot_df.empty:
        print("No rows available for requested history plot.")
        return
    fig, axes = plt.subplots(len(grains), 1, figsize=(14, 3.2 * len(grains)), sharex=True)
    axes = np.atleast_1d(axes)
    for axis, grain in zip(axes, grains):
        grain_df = plot_df[plot_df["grain"].eq(grain)]
        for state, state_df in grain_df.groupby("state_name"):
            axis.plot(state_df["date"], state_df["price"], linewidth=1.4, label=state)
        axis.set_title(f"{grain} price trend")
        axis.set_ylabel("Price")
        axis.grid(alpha=0.25)
        axis.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


def plot_recent_history(canonical: pd.DataFrame, days: int = 365, state: str = "All States") -> None:
    print_section(f"Recent {days}-Day Trend: {state}")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    cutoff = frame["date"].max() - pd.Timedelta(days=days)
    frame = frame[frame["date"].ge(cutoff) & frame["state_name"].eq(state)]
    if frame.empty:
        print("No recent rows available.")
        return
    pivot = frame.pivot_table(index="date", columns="grain", values="price", aggfunc="median").sort_index()
    pivot.plot(figsize=(14, 5), linewidth=1.7, title=f"{state}: recent price trend")
    plt.grid(alpha=0.25)
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()


def training_summary(registry: dict) -> pd.DataFrame:
    print_section("Training / Model Selection Summary")
    rows = []
    for grain, horizons in registry.get("models", {}).items():
        for horizon, trained in horizons.items():
            gates = trained.get("gates", {})
            method_counts = pd.Series([gate.get("selected_method", "unknown") for gate in gates.values()]).value_counts().to_dict()
            mapes = [
                gate.get("ml_mape")
                for gate in gates.values()
                if gate.get("ml_mape") is not None and np.isfinite(gate.get("ml_mape"))
            ]
            maes = [
                gate.get("ml_mae")
                for gate in gates.values()
                if gate.get("ml_mae") is not None and np.isfinite(gate.get("ml_mae"))
            ]
            rows.append({
                "grain": grain,
                "horizon": int(horizon),
                "candidate_models": ", ".join(trained.get("models", {}).keys()),
                "states": len(gates),
                "method_counts": method_counts,
                "median_selected_mape": round(float(np.median(mapes)), 3) if mapes else None,
                "mean_selected_mape": round(float(np.mean(mapes)), 3) if mapes else None,
                "median_selected_mae": round(float(np.median(maes)), 2) if maes else None,
                "mean_selected_mae": round(float(np.mean(maes)), 2) if maes else None,
            })
    summary = pd.DataFrame(rows).sort_values(["grain", "horizon"]) if rows else pd.DataFrame()
    display_frame(summary, rows=100)
    return summary


def method_leaderboard(registry: dict, top: int = 40) -> pd.DataFrame:
    print_section("Per-State Method Leaderboard")
    rows = []
    for grain, horizons in registry.get("models", {}).items():
        for horizon, trained in horizons.items():
            for state, gate in trained.get("gates", {}).items():
                method_mapes = gate.get("method_mapes") or {}
                method_maes = gate.get("method_maes") or {}
                best_method = min(method_mapes, key=method_mapes.get) if method_mapes else gate.get("selected_method")
                rows.append({
                    "grain": grain,
                    "horizon": int(horizon),
                    "state": state,
                    "selected_method": gate.get("selected_method"),
                    "best_method": best_method,
                    "selected_mape": gate.get("ml_mape"),
                    "selected_mae": gate.get("ml_mae"),
                    "baseline_mape": gate.get("baseline_mape"),
                    "baseline_mae": gate.get("baseline_mae"),
                    "sample_count": gate.get("sample_count"),
                    "all_method_mapes": method_mapes,
                    "all_method_maes": method_maes,
                })
    leaderboard = pd.DataFrame(rows)
    if leaderboard.empty:
        print("No leaderboard rows available.")
        return leaderboard
    leaderboard = leaderboard.sort_values(["grain", "horizon", "selected_mape"], na_position="last")
    display_frame(leaderboard, rows=top)
    return leaderboard


def validation_rows_summary(registry: dict) -> pd.DataFrame:
    print_section("Validation Rows Summary")
    rows = pd.DataFrame(registry.get("validation_rows", []))
    if rows.empty:
        print("No validation rows available.")
        return rows
    rows["error_pct"] = (rows["predicted_price"] - rows["actual_price"]).abs() / rows["actual_price"].replace(0, np.nan) * 100
    rows["abs_error"] = (rows["predicted_price"] - rows["actual_price"]).abs()
    summary = (
        rows.groupby(["grain", "horizon", "method"])
        .agg(rows=("actual_price", "size"), mape=("error_pct", "mean"), mae=("abs_error", "mean"))
        .reset_index()
    )
    summary["mape"] = summary["mape"].round(3)
    summary["mae"] = summary["mae"].round(2)
    display_frame(summary.sort_values(["grain", "horizon", "mape"]), rows=100)
    return rows


def plot_validation_fit(registry: dict, grain: str = "Wheat", state: str = "All States", horizon: int = 7) -> None:
    print_section(f"Validation Fit Plot: {grain} / {state} / {horizon}d")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    rows = pd.DataFrame(registry.get("validation_rows", []))
    if rows.empty:
        print("No validation rows available.")
        return
    rows = rows[
        rows["grain"].eq(grain)
        & rows["state_name"].eq(state)
        & rows["horizon"].eq(horizon)
    ].copy()
    if rows.empty:
        print("No matching validation rows.")
        return
    rows["target_date"] = pd.to_datetime(rows["target_date"])
    rows = rows.sort_values("target_date")
    plt.figure(figsize=(14, 5))
    plt.plot(rows["target_date"], rows["actual_price"], label="Actual", linewidth=2)
    plt.plot(rows["target_date"], rows["predicted_price"], label="Predicted", linewidth=1.7)
    plt.title(f"{grain} {state} {horizon}d validation fit")
    plt.ylabel("Price")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def efficiency_summary(efficiency: dict) -> pd.DataFrame:
    print_section("Historical Efficiency / Backtest Summary")
    rows = []
    for grain, states in efficiency.items():
        for state, horizons in states.items():
            for horizon, payload in horizons.items():
                metric = payload.get("metrics", {})
                rows.append({
                    "grain": grain,
                    "state": state,
                    "horizon": int(horizon),
                    "sample_count": metric.get("sample_count"),
                    "mape": metric.get("mape"),
                    "mae": metric.get("mae"),
                    "rmse": metric.get("rmse"),
                    "wape": metric.get("wape"),
                })
    summary = pd.DataFrame(rows)
    if summary.empty:
        print("No efficiency rows available.")
        return summary
    display_frame(summary.sort_values(["grain", "horizon", "mape"], na_position="last"), rows=100)
    return summary


def plot_efficiency_series(efficiency: dict, grain: str = "Wheat", state: str = "All States", horizon: int = 7, tail: int = 365) -> None:
    print_section(f"Efficiency Series Plot: {grain} / {state} / {horizon}d")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    payload = efficiency.get(grain, {}).get(state, {}).get(str(horizon), {})
    series = pd.DataFrame(payload.get("series", []))
    if series.empty:
        print("No efficiency series for this selection.")
        return
    series["date"] = pd.to_datetime(series["date"])
    series = series.sort_values("date").tail(tail)
    plt.figure(figsize=(14, 5))
    plt.plot(series["date"], series["actual_price"], label="Actual", linewidth=2)
    plt.plot(series["date"], series["predicted_price"], label="Predicted", linewidth=1.5)
    plt.title(f"{grain} {state} {horizon}d efficiency, last {tail} rows")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def release_file_summary() -> pd.DataFrame:
    print_section("Release File Validation")
    files = sorted(path for path in RELEASE_DIR.iterdir() if path.is_file())
    summary = pd.DataFrame([{
        "file": path.name,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
    } for path in files])
    display_frame(summary, rows=100)

    manifest_path = RELEASE_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        print("\nManifest core:")
        display_frame(pd.DataFrame([{
            "run_id": manifest.get("run_id"),
            "generated_at": manifest.get("generated_at"),
            "data_latest_date": manifest.get("data_latest_date"),
            "actuals_row_count": manifest.get("actuals_row_count"),
            "states": len(manifest.get("states", [])),
            "grains": ", ".join(manifest.get("grains", [])),
        }]))
    return summary


def inspect_prediction_output(grain: str = "Wheat", state: str = "All States") -> dict:
    print_section(f"Prediction Output Inspection: {grain} / {state}")
    path = RELEASE_DIR / "predictions.json"
    if not path.exists():
        print("predictions.json not found.")
        return {}
    predictions = json.loads(path.read_text(encoding="utf-8"))
    payload = predictions.get(grain, {}).get(state, {})
    print(json.dumps(payload, indent=2)[:4000])
    return payload



def file_inventory(files: list[Path]) -> pd.DataFrame:
    print_section("Input File Inventory")
    rows = []
    for path in files:
        try:
            stat = path.stat()
            rows.append({
                "file": path.name,
                "extension": path.suffix.lower(),
                "size_mb": round(stat.st_size / (1024 * 1024), 3),
                "priority": 2 if path.name.lower() == "latest_data.csv" else 1,
                "path": str(path),
            })
        except Exception as exc:
            rows.append({"file": path.name, "extension": path.suffix.lower(), "size_mb": None, "priority": None, "path": str(path), "error": str(exc)})
    inventory = pd.DataFrame(rows)
    if inventory.empty:
        print(f"No CSV or Parquet inputs found under {INPUT_ROOT}")
    else:
        display_frame(inventory.sort_values(["priority", "file"], ascending=[False, True]), rows=max(len(inventory), 20))
    return inventory


def plot_file_inventory(inventory: pd.DataFrame, top: int = 30) -> None:
    print_section("Input File Size Visualization")
    if plt is None or inventory.empty or "size_mb" not in inventory:
        print("No file-size chart available.")
        return
    plot_df = inventory.dropna(subset=["size_mb"]).nlargest(top, "size_mb").sort_values("size_mb")
    if plot_df.empty:
        print("No readable file sizes.")
        return
    plt.figure(figsize=(12, max(4, len(plot_df) * 0.35)))
    plt.barh(plot_df["file"], plot_df["size_mb"])
    plt.xlabel("Size (MB)")
    plt.title(f"Largest {len(plot_df)} discovered input files")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


def preview_input_file(path: Path, rows: int = 5) -> pd.DataFrame:
    print_section(f"Schema Preview: {path.name}")
    try:
        if path.suffix.lower() == ".csv":
            sample = pd.read_csv(path, nrows=rows)
        else:
            try:
                import pyarrow.parquet as pq
                parquet_file = pq.ParquetFile(path)
                sample = parquet_file.read_row_group(0).slice(0, rows).to_pandas()
                print(f"Parquet row groups: {parquet_file.num_row_groups}; schema columns: {len(parquet_file.schema.names)}")
            except Exception:
                sample = pd.read_parquet(path).head(rows)
        mapping = resolve_columns(sample.columns)
        schema = pd.DataFrame({
            "canonical_field": list(mapping.keys()),
            "resolved_source_column": list(mapping.values()),
        })
        print("Resolved column mapping")
        display_frame(schema, rows=50)
        print("Sample rows")
        display_frame(sample, rows=rows)
        return sample
    except Exception as exc:
        print(f"Could not preview {path}: {type(exc).__name__}: {exc}")
        return pd.DataFrame()


def dataset_checkpoint(frame: pd.DataFrame, label: str, preview_rows: int = 8) -> pd.DataFrame:
    print_section(f"Dataset Checkpoint: {label}")
    if frame is None or frame.empty:
        print("No rows at this checkpoint.")
        return pd.DataFrame()
    df = frame.copy()
    if "date" in df:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    summary = pd.DataFrame([{
        "checkpoint": label,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "min_date": df["date"].min().date().isoformat() if "date" in df and df["date"].notna().any() else None,
        "max_date": df["date"].max().date().isoformat() if "date" in df and df["date"].notna().any() else None,
        "grains": int(df["grain"].nunique()) if "grain" in df else None,
        "states": int(df["state_name"].nunique()) if "state_name" in df else None,
        "duplicate_keys": int(df.duplicated(["date", "state_name", "grain"]).sum()) if {"date", "state_name", "grain"}.issubset(df.columns) else None,
        "price_missing_pct": round(float(pd.to_numeric(df.get("price"), errors="coerce").isna().mean() * 100), 2) if "price" in df else None,
    }])
    display_frame(summary)

    if "source" in df:
        source_mix = df.groupby("source", dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)
        source_mix["share_pct"] = (source_mix["rows"] / len(df) * 100).round(2)
        print("\nSource composition")
        display_frame(source_mix, rows=50)

    if {"grain", "price"}.issubset(df.columns):
        by_grain = df.groupby("grain").agg(
            rows=("price", "size"),
            states=("state_name", "nunique") if "state_name" in df else ("price", "size"),
            median_price=("price", "median"),
            min_price=("price", "min"),
            max_price=("price", "max"),
        ).reset_index()
        for col in ["median_price", "min_price", "max_price"]:
            by_grain[col] = pd.to_numeric(by_grain[col], errors="coerce").round(2)
        print("\nGrain-level profile")
        display_frame(by_grain, rows=50)

    print("\nFirst rows")
    display_frame(df.sort_values("date").head(preview_rows) if "date" in df else df.head(preview_rows), rows=preview_rows)
    print("\nLatest rows")
    display_frame(df.sort_values("date", ascending=False).head(preview_rows) if "date" in df else df.tail(preview_rows), rows=preview_rows)
    return summary


def merge_audit(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    print_section("Priority Merge Audit - Before Deduplication")
    rows = []
    combined_parts = []
    for label, frame in frames.items():
        if frame is None or frame.empty:
            rows.append({"input": label, "rows": 0, "unique_keys": 0, "duplicate_keys_inside_input": 0, "min_date": None, "max_date": None})
            continue
        aligned = align_canonical(frame)
        keys = ["date", "state_name", "grain"]
        rows.append({
            "input": label,
            "rows": int(len(aligned)),
            "unique_keys": int(aligned[keys].drop_duplicates().shape[0]),
            "duplicate_keys_inside_input": int(aligned.duplicated(keys).sum()),
            "min_date": str(aligned["date"].min()),
            "max_date": str(aligned["date"].max()),
        })
        temp = aligned.copy()
        temp["audit_input"] = label
        combined_parts.append(temp)
    audit = pd.DataFrame(rows)
    display_frame(audit, rows=50)
    if combined_parts:
        combined = pd.concat(combined_parts, ignore_index=True)
        key_counts = combined.groupby(["date", "state_name", "grain"]).size()
        overlap = int((key_counts > 1).sum())
        removable = int((key_counts - 1).clip(lower=0).sum())
        print(f"Overlapping keys across sources: {overlap:,}")
        print(f"Rows expected to be removed by source-priority deduplication: {removable:,}")
    return audit


def plot_monthly_coverage(canonical: pd.DataFrame, state: str = "All States") -> None:
    print_section(f"Monthly Data Coverage Heatmap: {state}")
    if plt is None or canonical.empty:
        print("Coverage plot unavailable.")
        return
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame = frame[frame["state_name"].eq(state)].dropna(subset=["date"])
    if frame.empty:
        print(f"No rows for {state}.")
        return
    frame["month"] = frame["date"].dt.to_period("M").astype(str)
    coverage = frame.pivot_table(index="grain", columns="month", values="price", aggfunc="count", fill_value=0)
    # Keep the most recent 72 months legible; the table below still reports full range.
    visible = coverage.iloc[:, -72:]
    plt.figure(figsize=(16, max(3, len(visible) * 1.1)))
    image = plt.imshow(visible.values, aspect="auto", interpolation="nearest")
    plt.colorbar(image, label="Rows in month")
    plt.yticks(range(len(visible.index)), visible.index)
    tick_positions = np.linspace(0, max(len(visible.columns) - 1, 0), min(12, len(visible.columns)), dtype=int) if len(visible.columns) else []
    if len(tick_positions):
        plt.xticks(tick_positions, [visible.columns[i] for i in tick_positions], rotation=45, ha="right")
    plt.title(f"Monthly observation coverage - {state} (latest {visible.shape[1]} months)")
    plt.tight_layout()
    plt.show()


def plot_price_distribution(canonical: pd.DataFrame, state: str = "All States") -> None:
    print_section(f"Price Distribution: {state}")
    if plt is None or canonical.empty:
        print("Price distribution plot unavailable.")
        return
    frame = canonical[canonical["state_name"].eq(state)].copy()
    if frame.empty:
        print(f"No rows for {state}.")
        return
    data = []
    labels = []
    for grain in TARGET_GRAINS:
        values = pd.to_numeric(frame.loc[frame["grain"].eq(grain), "price"], errors="coerce").dropna()
        if not values.empty:
            low, high = values.quantile([0.01, 0.99])
            data.append(values.clip(low, high).to_numpy())
            labels.append(grain)
    if not data:
        print("No numeric price values.")
        return
    plt.figure(figsize=(11, 5))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.ylabel("Price")
    plt.title(f"Price distributions by grain - {state} (1st-99th percentile clipped)")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_arrival_price_relationship(canonical: pd.DataFrame, state: str = "All States", sample_rows: int = 6000) -> None:
    print_section(f"Arrival vs Price Relationship: {state}")
    if plt is None or canonical.empty or "arrival" not in canonical:
        print("Arrival relationship plot unavailable.")
        return
    frame = canonical[canonical["state_name"].eq(state)].copy()
    frame["arrival"] = pd.to_numeric(frame["arrival"], errors="coerce")
    frame["price"] = pd.to_numeric(frame["price"], errors="coerce")
    frame = frame.dropna(subset=["arrival", "price"])
    frame = frame[frame["arrival"].gt(0)]
    if frame.empty:
        print("No positive arrival rows available.")
        return
    if len(frame) > sample_rows:
        frame = frame.sample(sample_rows, random_state=42)
    plt.figure(figsize=(11, 5))
    for grain, group in frame.groupby("grain"):
        plt.scatter(np.log1p(group["arrival"]), group["price"], s=12, alpha=0.35, label=grain)
    plt.xlabel("log(1 + arrival)")
    plt.ylabel("Price")
    plt.title(f"Arrival-price relationship - {state}")
    plt.grid(alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.show()


def feature_checkpoint(features: pd.DataFrame) -> pd.DataFrame:
    print_section("Feature Engineering Checkpoint")
    if features is None or features.empty:
        print("Feature table is empty.")
        return pd.DataFrame()
    feature_cols = [col for col in FEATURE_COLUMNS if col in features.columns]
    summary = pd.DataFrame([{
        "rows": int(len(features)),
        "total_columns": int(features.shape[1]),
        "model_features_expected": int(len(FEATURE_COLUMNS)),
        "model_features_present": int(len(feature_cols)),
        "missing_expected_features": ", ".join(sorted(set(FEATURE_COLUMNS) - set(feature_cols))) or "None",
        "date_min": pd.to_datetime(features["date"]).min().date().isoformat(),
        "date_max": pd.to_datetime(features["date"]).max().date().isoformat(),
    }])
    display_frame(summary)

    missing = pd.DataFrame({
        "feature": feature_cols,
        "missing_pct": [round(float(features[col].isna().mean() * 100), 2) for col in feature_cols],
        "non_null_rows": [int(features[col].notna().sum()) for col in feature_cols],
        "median": [round(float(pd.to_numeric(features[col], errors="coerce").median()), 4) if pd.to_numeric(features[col], errors="coerce").notna().any() else None for col in feature_cols],
    }).sort_values("missing_pct", ascending=False)
    print("\nFeature availability")
    display_frame(missing, rows=len(missing))

    if plt is not None and not missing.empty:
        visible = missing.head(30).sort_values("missing_pct")
        plt.figure(figsize=(11, max(5, len(visible) * 0.28)))
        plt.barh(visible["feature"], visible["missing_pct"])
        plt.xlabel("Missing values (%)")
        plt.title("30 model features with the most missing values")
        plt.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()

    sample_cols = ["date", "grain", "state_name", "price"] + [c for c in ["price_lag_1", "price_lag_7", "rolling_mean_30", "arrival_rolling_mean_30", "national_price", "state_national_spread"] if c in features]
    print("\nFeature-row sample")
    display_frame(features[sample_cols].sort_values("date", ascending=False), rows=15)
    return missing


def plot_feature_relationships(features: pd.DataFrame, grain: str = "Wheat", state: str = "All States", days: int = 365) -> None:
    print_section(f"Feature Relationship Plot: {grain} / {state}")
    if plt is None or features.empty:
        print("Feature plot unavailable.")
        return
    frame = features[(features["grain"].eq(grain)) & (features["state_name"].eq(state))].copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.sort_values("date")
    if frame.empty:
        print("No matching feature rows.")
        return
    cutoff = frame["date"].max() - pd.Timedelta(days=days)
    frame = frame[frame["date"].ge(cutoff)]
    plt.figure(figsize=(14, 5))
    plt.plot(frame["date"], frame["price"], label="Price", linewidth=2)
    for col, label in [("rolling_mean_7", "7d rolling mean"), ("rolling_mean_30", "30d rolling mean"), ("national_price", "National price")]:
        if col in frame:
            plt.plot(frame["date"], frame[col], label=label, linewidth=1.4)
    plt.ylabel("Price / feature value")
    plt.title(f"Engineered price features - {grain}, {state}")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def training_slot_snapshot(trained: dict, tail: int = 180) -> pd.DataFrame:
    grain = trained.get("grain")
    horizon = int(trained.get("horizon"))
    print_section(f"Immediate Training Result: {grain} / {horizon}d")
    mapes = trained.get("global_method_mapes", {})
    maes = trained.get("global_method_mae", {})
    score_table = pd.DataFrame({
        "method": list(mapes.keys()),
        "global_mape_pct": list(mapes.values()),
        "global_mae": [maes.get(name) for name in mapes],
    }).sort_values(["global_mape_pct", "global_mae"]) if mapes else pd.DataFrame()
    if not score_table.empty:
        display_frame(score_table, rows=50)
    split = trained.get("split_info", {})
    print("Split:", split)
    method_counts = pd.Series([gate.get("selected_method", "unknown") for gate in trained.get("gates", {}).values()]).value_counts().rename_axis("selected_method").reset_index(name="states")
    print("\nSelected method by state")
    display_frame(method_counts, rows=50)

    rows = pd.DataFrame(trained.get("validation_rows", []))
    if plt is None or rows.empty:
        return score_table
    national = rows[rows["state_name"].eq("All States")].copy()
    if national.empty:
        national = rows.copy()
    national["target_date"] = pd.to_datetime(national["target_date"])
    national = national.sort_values("target_date").tail(tail)
    plt.figure(figsize=(14, 4.8))
    plt.plot(national["target_date"], national["actual_price"], label="Actual", linewidth=2)
    plt.plot(national["target_date"], national["predicted_price"], label="Selected prediction", linewidth=1.6)
    shown_state = "All States" if (rows["state_name"] == "All States").any() else "all validation rows"
    plt.title(f"Immediate validation fit - {grain}, {shown_state}, {horizon}d")
    plt.ylabel("Price")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
    return score_table


def forecast_checkpoint(predictions: dict, forecast_series: dict, grain: str = "Wheat", state: str = "All States") -> pd.DataFrame:
    print_section(f"Forecast Checkpoint: {grain} / {state}")
    payload = predictions.get(grain, {}).get(state, {})
    if not payload:
        print("No forecast payload for this grain/state.")
        return pd.DataFrame()
    rows = [{
        "horizon_days": int(horizon),
        "target_date": details.get("target_date"),
        "predicted_price": details.get("predicted_price"),
        "selected_method": details.get("selected_method"),
        "mape_pct": (details.get("metrics") or {}).get("mape"),
        "mae": (details.get("metrics") or {}).get("mae"),
        "baseline_mape_pct": (details.get("metrics") or {}).get("baseline_mape"),
        "baseline_mae": (details.get("metrics") or {}).get("baseline_mae"),
        "interval_lower": (details.get("prediction_interval") or {}).get("lower"),
        "interval_upper": (details.get("prediction_interval") or {}).get("upper"),
        "selection_reason": details.get("selection_reason"),
        "validation_samples": (details.get("metrics") or {}).get("sample_count"),
    } for horizon, details in payload.get("horizons", {}).items()]
    table = pd.DataFrame(rows).sort_values("horizon_days")
    print(
        f"Last actual date: {payload.get('last_actual_date')}; current price: {payload.get('current_price')}; "
        f"freshness={payload.get('status')} ({payload.get('days_since_last_actual', 0)} days behind dataset max)"
    )
    display_frame(table, rows=50)

    series = pd.DataFrame(forecast_series.get(grain, {}).get(state, []))
    if plt is not None and not series.empty:
        series["date"] = pd.to_datetime(series["date"])
        plt.figure(figsize=(14, 4.8))
        plt.plot(series["date"], series["price"], label="Interpolated daily forecast", linewidth=1.8)
        anchors = series[series["is_anchor"].fillna(False)]
        if not anchors.empty:
            plt.scatter(anchors["date"], anchors["price"], s=55, label="Model horizon anchors")
        plt.axhline(float(payload.get("current_price")), linestyle="--", linewidth=1.2, label="Current price")
        plt.title(f"Future forecast path - {grain}, {state}")
        plt.ylabel("Price")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()
    return table


def reasoning_checkpoint(reasoning: dict, grain: str = "Wheat", state: str = "All States") -> dict:
    print_section(f"Reasoning Checkpoint: {grain} / {state}")
    payload = reasoning.get(grain, {}).get(state, {})
    if not payload:
        print("No reasoning payload available.")
        return {}
    print(json.dumps(payload, indent=2, ensure_ascii=False)[:6000])
    return payload


## 0A. Live Price Sanity Check Before Training

This runs before historical loading and model training. If these prices do not match the live Market Wise Price and Arrival table, stop here and fix the data/cache before waiting for the full training run.


In [ ]:
print("\nLive dashboard price sanity check before training")
print("If these do not match the website Market Wise Price and Arrival table, stop the run here.")

live_price_check = fetch_live_dashboard_prices_from_supabase()
if not live_price_check:
    print("No live dashboard price overrides found. Training can continue, but current_price will come from canonical data only.")
else:
    live_price_rows = []
    for grain in TARGET_GRAINS:
        payload = live_price_check.get(grain)
        live_price_rows.append({
            "grain": grain,
            "live_price": None if not payload else payload.get("price"),
            "reported_dates": None if not payload else payload.get("reported_dates"),
            "fetched_at": None if not payload else payload.get("fetched_at"),
            "cache_key": None if not payload else payload.get("cache_key"),
            "source": None if not payload else payload.get("source"),
        })
    live_price_frame = pd.DataFrame(live_price_rows)
    display(live_price_frame)
    missing_live_prices = live_price_frame[live_price_frame["live_price"].isna()]["grain"].tolist()
    if missing_live_prices:
        print("Missing live prices for:", missing_live_prices)
    else:
        print("All target grain live prices were found before training.")


## 1. Discover Every Input File Before Loading Data

This stage lists the exact files selected by the notebook, shows their size and source priority, and previews source-column mappings before any aggregation occurs.

In [ ]:
input_files = discover_data_files()
input_inventory = file_inventory(input_files)
plot_file_inventory(input_inventory)

files_to_preview = input_files[:TRANSPARENCY_MAX_SCHEMA_FILES]
print(f"Previewing schemas for {len(files_to_preview)} of {len(input_files)} discovered files.")
for input_path in files_to_preview:
    preview_input_file(input_path, rows=min(5, TRANSPARENCY_PREVIEW_ROWS))

## 2. Inspect the Previous Canonical Snapshot

The previous snapshot is loaded separately. You can verify whether incremental mode is active and exactly what historical range is already available.

In [ ]:
previous = load_previous_canonical()
if previous.empty:
    print("No previous canonical snapshot is available. A historical rebuild will be attempted.")
else:
    print("Previous canonical snapshot loaded successfully.")
dataset_checkpoint(previous, "Previous canonical snapshot", preview_rows=TRANSPARENCY_PREVIEW_ROWS)

## 3. Load and Normalize Historical Files

Historical files are loaded only when required by the existing incremental-rebuild policy. The normalized historical table is inspected immediately.

In [ ]:
should_load_historical = previous.empty or FORCE_FULL_REBUILD
print(f"Historical loading required: {should_load_historical}")
if should_load_historical:
    historical = load_historical_sources()
else:
    historical = pd.DataFrame()
    print("Historical file reload skipped because a previous canonical snapshot exists and FORCE_FULL_REBUILD is false.")
dataset_checkpoint(historical, "Normalized and daily-aggregated historical data", preview_rows=TRANSPARENCY_PREVIEW_ROWS)

## 4. Fetch and Inspect New Supabase Actuals

Only rows newer than the previous canonical maximum date are requested. The returned delta is inspected before it can affect the canonical dataset.

In [ ]:
previous_latest_date = None if previous.empty else str(previous["date"].max())
print(f"Supabase lower-bound date: {previous_latest_date or 'None - fetch all available rows'}")
supabase_delta = fetch_supabase_actuals(previous_latest_date)
dataset_checkpoint(supabase_delta, "Supabase incremental actuals", preview_rows=TRANSPARENCY_PREVIEW_ROWS)

## 5. Audit the Priority Merge, Validate, and Save Canonical Data

Source overlap and expected deduplication are shown before the merge. After merging, validation runs before files are written.

In [ ]:
merge_inputs = {
    "historical": historical,
    "previous_canonical": previous,
    "supabase_delta": supabase_delta,
}
merge_audit(merge_inputs)

canonical = merge_priority([historical, previous, supabase_delta])
print(f"Rows after priority merge: {len(canonical):,}")
validate_canonical(canonical)
print("Canonical validation passed: non-empty, unique date/state/grain keys, positive prices, and All States rows present.")
save_canonical(canonical)
print(f"Canonical files written to {CANONICAL_PARQUET} and {CANONICAL_CSV}")
dataset_checkpoint(canonical, "Final canonical dataset", preview_rows=TRANSPARENCY_PREVIEW_ROWS)

## 6. Visual Data Audit Before Feature Engineering

These checks expose coverage gaps, missing values, recent additions, long-run trends, price distributions, and arrival-price relationships before model features are created.

In [ ]:
canonical_profile = canonical_summary(canonical)
canonical_missingness = missingness_summary(canonical)
canonical_recent = recent_data_summary(canonical, days=14)

In [ ]:
plot_history(canonical, states=["All States"], grains=TARGET_GRAINS)
plot_recent_history(canonical, days=365, state="All States")
plot_monthly_coverage(canonical, state="All States")
plot_price_distribution(canonical, state="All States")
plot_arrival_price_relationship(canonical, state="All States")

## 7. Engineer Features and Inspect Them Before Training

The feature matrix is built in its own step, so lag availability, rolling features, national context, and missingness can be checked before any model sees the data.

In [ ]:
features = feature_engineering(canonical)
print(f"Engineered feature table: {features.shape[0]:,} rows x {features.shape[1]:,} columns")
feature_missingness = feature_checkpoint(features)
plot_feature_relationships(features, grain="Wheat", state="All States", days=365)

## 8. Train Leakage-Safe Ensemble Models

Each grain and horizon uses chronological train, ensemble-calibration, and final-holdout windows. Candidate models are promoted only when they beat persistence across temporal folds. Production models are refit only after evaluation is frozen.


In [ ]:
registry = train_models(canonical, features=features, transparent=TRANSPARENT_MODE)

## 9. Review the Complete Model Registry

This consolidated view is shown after the live per-slot outputs, not instead of them.

In [ ]:
registry_summary = training_summary(registry)
registry_leaderboard = method_leaderboard(registry, top=120)
validation_rows = validation_rows_summary(registry)

In [ ]:
for horizon in HORIZONS:
    plot_validation_fit(registry, grain="Wheat", state="All States", horizon=horizon)

## 10. Generate Forecasts and Inspect Them Immediately

The website-compatible files are written first, then each All States forecast is summarized and plotted as a daily path with model horizon anchors.

In [ ]:
predictions, forecast_series, actuals, metrics = generate_predictions(canonical, registry)
print("Prediction grains written:", list(predictions.keys()))
for grain in TARGET_GRAINS:
    forecast_checkpoint(predictions, forecast_series, grain=grain, state="All States")

In [ ]:
inspect_prediction_output("Wheat", "All States")

## 11. Generate Genuine Holdout Efficiency and Backtest Visualizations

Only embargoed holdout predictions are published here. The old multi-year persistence rows are no longer presented as model backtests, and each series is capped to keep the website payload practical.


In [ ]:
efficiency, backtest = generate_efficiency_data(registry)
efficiency_table = efficiency_summary(efficiency)
for horizon in HORIZONS:
    plot_efficiency_series(
        efficiency,
        grain="Wheat",
        state="All States",
        horizon=horizon,
        tail=min(365, max(120, TRANSPARENCY_VALIDATION_PLOT_TAIL)),
    )

In [ ]:
evaluation_report = generate_evaluation_report(registry)
data_drift_report = generate_data_drift_report(canonical)
print("Evaluation strategy:", evaluation_report.get("evaluation_strategy"))
print("National holdout MAPE range:", evaluation_report.get("summary", {}).get("national_min_mape"), "to", evaluation_report.get("summary", {}).get("national_max_mape"))
print("Data drift warnings:", data_drift_report.get("summary", {}).get("warning_series"), "/", data_drift_report.get("summary", {}).get("series_checked"))
display(pd.DataFrame(evaluation_report.get("national_metrics", [])))


## 12. Generate and Inspect Model-Derived Reasoning

Driver weights now come from local model perturbations rather than a fixed hardcoded list. Reasoning also includes calibrated intervals, validation evidence, and stale-actual warnings.


In [ ]:
reasoning = generate_reasoning(predictions, metrics)
print("Reasoning generated for grains:", list(reasoning.keys()))
for grain in TARGET_GRAINS:
    reasoning_checkpoint(reasoning, grain=grain, state="All States")

## Mandi-Level Forecast Extension

Generated profile: **showcase**.

This section keeps the existing state-wise website contract unchanged and adds optional mandi-level files:

- `markets.json`: available mandi/market series with state, district, latest date, and optional coordinates.
- `market_predictions.json`: mandi-wise current price and 7/30/90 day forecasts. User-specific carrying charges and farm-gate prices are added by the backend after distance is known.
- `market_forecast_series.json`: mandi-wise chart forecast paths.
- `market_actuals.json`: recent mandi-wise actual context.
- `market_reasoning.json`: lightweight market notes; the backend can still enrich reasoning with Gemini.

Nearest-mandi distance works only when market coordinates are available. If the source data does not include coordinates, the website safely falls back to selected/state forecast until a `mandi_locations.csv` input is attached.


In [ ]:
import hashlib

ENABLE_MANDI_LEVEL_RELEASE = env_bool("ENABLE_MANDI_LEVEL_RELEASE", True)
ENABLE_MANDI_LEVEL_FULL_TRAINING = env_bool("ENABLE_MANDI_LEVEL_FULL_TRAINING", False)
MAX_MARKET_SERIES = int(os.environ.get("MAX_MARKET_SERIES", "0"))
MIN_MARKET_OBSERVED_DAYS = int(os.environ.get("MIN_MARKET_OBSERVED_DAYS", "180"))
MANDI_FORECAST_HISTORY_DAYS = int(os.environ.get("MANDI_FORECAST_HISTORY_DAYS", "60"))
CONFORMAL_ALPHA = float(os.environ.get("CONFORMAL_ALPHA", "0.10"))
HORIZON_PRICE_CLIP_BOUNDS = {7: (0.70, 1.35), 30: (0.60, 1.55), 90: (0.50, 1.85)}

MARKET_LOCATION_ALIASES = {
    "state_name": ["state", "state_name", "state_name_en"],
    "district": ["district", "district_name"],
    "market": ["market", "market_name", "mandi", "mandi_name"],
    "lat": ["lat", "latitude"],
    "lng": ["lng", "lon", "longitude"],
}


def _slug(value):
    text = str(value or "").strip().lower()
    slug = "".join(ch if ch.isalnum() else "-" for ch in text)
    return "-".join(part for part in slug.split("-") if part)


def market_key_for(row):
    raw = "::".join(str(row.get(col, "") or "").strip() for col in ["state_name", "district", "market"])
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:10]
    return f"{_slug(raw)[:80]}-{digest}"


def interpolate_market_series(last_date, current_price, horizon_points, horizon_intervals):
    points = {0: float(current_price), **{int(key): float(value) for key, value in horizon_points.items()}}
    lower_points = {0: float(current_price), **{int(key): float(value[0]) for key, value in horizon_intervals.items()}}
    upper_points = {0: float(current_price), **{int(key): float(value[1]) for key, value in horizon_intervals.items()}}
    xs = np.array(sorted(points), dtype=float)
    prices = np.array([points[int(x)] for x in xs], dtype=float)
    lowers = np.array([lower_points.get(int(x), points[int(x)]) for x in xs], dtype=float)
    uppers = np.array([upper_points.get(int(x), points[int(x)]) for x in xs], dtype=float)
    output = []
    for day in range(1, int(xs.max()) + 1):
        price = float(np.interp(day, xs, prices))
        lower = min(price, float(np.interp(day, xs, lowers)))
        upper = max(price, float(np.interp(day, xs, uppers)))
        output.append({
            "date": (pd.to_datetime(last_date) + pd.Timedelta(days=day)).date().isoformat(),
            "price": round(price, 2),
            "confidence_lower": round(lower, 2),
            "confidence_upper": round(upper, 2),
            "is_anchor": day in horizon_points,
            "anchor_horizon": day if day in horizon_points else None,
        })
    return output


def _resolve_location_columns(columns):
    by_lower = {str(column).lower().strip(): column for column in columns}
    return {
        target: next((by_lower[alias] for alias in aliases if alias in by_lower), None)
        for target, aliases in MARKET_LOCATION_ALIASES.items()
    }


def load_market_locations():
    frames = []
    for input_path in INPUT_ROOT.rglob("*"):
        if not input_path.is_file() or input_path.suffix.lower() not in {".csv", ".parquet"}:
            continue
        if "location" not in input_path.name.lower() and "mandi" not in input_path.name.lower() and "market" not in input_path.name.lower():
            continue
        try:
            sample = pd.read_parquet(input_path) if input_path.suffix.lower() == ".parquet" else pd.read_csv(input_path)
        except Exception:
            continue
        resolved = _resolve_location_columns(sample.columns)
        if not resolved["market"] or not resolved["lat"] or not resolved["lng"]:
            continue
        out = pd.DataFrame({
            "state_name": sample[resolved["state_name"]] if resolved["state_name"] else pd.NA,
            "district": sample[resolved["district"]] if resolved["district"] else pd.NA,
            "market": sample[resolved["market"]],
            "lat": pd.to_numeric(sample[resolved["lat"]], errors="coerce"),
            "lng": pd.to_numeric(sample[resolved["lng"]], errors="coerce"),
        }).dropna(subset=["market", "lat", "lng"])
        if not out.empty:
            frames.append(out)
            print(f"Loaded mandi coordinates from {input_path}")
    if not frames:
        return pd.DataFrame(columns=["market_key", "lat", "lng"])
    locations = pd.concat(frames, ignore_index=True)
    locations["state_name"] = locations["state_name"].astype("string").str.strip()
    locations["district"] = locations["district"].astype("string").str.strip()
    locations["market"] = locations["market"].astype("string").str.strip()
    locations["market_key"] = locations.apply(market_key_for, axis=1)
    return locations.drop_duplicates("market_key", keep="last")[["market_key", "lat", "lng"]]


def load_market_level_sources():
    frames = []
    for input_path in discover_data_files():
        try:
            raw = pd.read_parquet(input_path) if input_path.suffix.lower() == ".parquet" else pd.read_csv(input_path)
        except Exception as exc:
            print(f"Skipping market-level load for {input_path}: {exc}")
            continue
        normalized = normalize_frame(raw, source=("latest_csv" if input_path.name.lower() == "latest_data.csv" else "historical"), source_priority=(2 if input_path.name.lower() == "latest_data.csv" else 1))
        if normalized.empty or "market" not in normalized.columns:
            continue
        normalized["district"] = normalized.get("district", pd.NA).astype("string").str.strip()
        normalized["market"] = normalized.get("market", pd.NA).astype("string").str.strip()
        normalized = normalized[normalized["market"].notna() & normalized["market"].ne("")]
        if not normalized.empty:
            frames.append(normalized)
    if not frames:
        return pd.DataFrame()

    raw_market = pd.concat(frames, ignore_index=True)
    price_agg = "mean" if AGGREGATION_METHOD == "mean" else "median"
    market_daily = raw_market.groupby(["date", "state_name", "district", "market", "grain"], as_index=False, observed=True).agg(
        price=("price", price_agg),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_count", "sum"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )
    market_daily["market_key"] = market_daily.apply(market_key_for, axis=1)
    market_daily["market_label"] = market_daily[["state_name", "district", "market"]].fillna("").agg(" / ".join, axis=1).str.replace(r"\s+/\s+/\s+", " / ", regex=True)

    coverage = market_daily.groupby(["market_key", "grain"], observed=True).agg(
        observed_days=("date", "nunique"),
        latest_date=("date", "max"),
        state_name=("state_name", "last"),
        district=("district", "last"),
        market=("market", "last"),
        row_count=("price", "size"),
    ).reset_index()
    eligible = coverage[coverage["observed_days"].ge(MIN_MARKET_OBSERVED_DAYS)].sort_values(["latest_date", "observed_days"], ascending=[False, False])
    selected = eligible["market_key"].drop_duplicates()
    if MAX_MARKET_SERIES > 0:
        selected = selected.head(MAX_MARKET_SERIES)
    selected_keys = selected.tolist()
    market_daily = market_daily[market_daily["market_key"].isin(selected_keys)].copy()

    locations = load_market_locations()
    if not locations.empty:
        market_daily = market_daily.merge(locations, on="market_key", how="left")
    else:
        market_daily["lat"] = pd.NA
        market_daily["lng"] = pd.NA

    print(f"Mandi-level daily rows: {len(market_daily):,}; selected mandi series: {len(selected_keys):,}")
    return market_daily


def build_market_training_canonical(market_daily):
    if market_daily.empty:
        return pd.DataFrame()
    out = market_daily.copy()
    out["original_state_name"] = out["state_name"]
    out["state_name"] = out["market_key"]
    out["state_id"] = out["market_key"]
    out["state_key"] = out["market_key"]

    all_markets = market_daily.groupby(["date", "grain"], as_index=False, observed=True).agg(
        price=("price", "median"),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_key", "nunique"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )
    all_markets["state_name"] = "All States"
    all_markets["state_id"] = "all-markets"
    all_markets["state_key"] = "all-markets"
    for col in ["district", "market", "market_key", "market_label", "original_state_name", "lat", "lng"]:
        all_markets[col] = pd.NA

    return align_canonical(pd.concat([out, all_markets[out.columns]], ignore_index=True))


def market_lookup_payload(market_daily):
    if market_daily.empty:
        return []
    grouped = market_daily.groupby("market_key", observed=True).agg(
        market_name=("market", "last"),
        district=("district", "last"),
        state=("state_name", "last"),
        latest_date=("date", "max"),
        row_count=("price", "size"),
        grain_count=("grain", "nunique"),
        lat=("lat", "last"),
        lng=("lng", "last"),
    ).reset_index()
    return [
        {
            "market_key": row.market_key,
            "market_name": row.market_name,
            "district": None if pd.isna(row.district) else row.district,
            "state": row.state,
            "latest_date": str(row.latest_date),
            "row_count": int(row.row_count),
            "grain_count": int(row.grain_count),
            "lat": None if pd.isna(row.lat) else float(row.lat),
            "lng": None if pd.isna(row.lng) else float(row.lng),
        }
        for row in grouped.itertuples(index=False)
    ]


def generate_market_predictions(market_daily, market_canonical, market_registry):
    market_predictions, market_forecast_series, market_actuals, market_metrics, market_reasoning = {}, {}, {}, {}, {}
    features = market_registry["features"].copy()
    features["date"] = pd.to_datetime(features["date"])
    latest_rows = features[~features["state_name"].eq("All States")].sort_values("date").groupby(["grain", "state_name"], as_index=False).tail(1)
    lookup = {item["market_key"]: item for item in market_lookup_payload(market_daily)}

    for grain in TARGET_GRAINS:
        market_predictions[grain] = {}
        market_forecast_series[grain] = {}
        market_actuals[grain] = {}
        market_metrics[grain] = {}
        market_reasoning[grain] = {}
        for _, row in latest_rows[latest_rows["grain"].eq(grain)].iterrows():
            market_key = row["state_name"]
            info = lookup.get(market_key, {"market_key": market_key})
            last_ts = pd.to_datetime(row["date"])
            last_date = last_ts.date()
            current_price = float(row["price"])
            horizon_points = {}
            horizon_intervals = {}
            prediction = {
                "current_price": round(current_price, 2),
                "last_actual_date": last_date.isoformat(),
                "forecast_start_date": (last_ts + pd.Timedelta(days=1)).date().isoformat(),
                "status": "fresh",
                "horizons": {},
            }
            market_metrics[grain][market_key] = {}
            for horizon in HORIZONS:
                trained = market_registry["models"].get(grain, {}).get(str(horizon))
                gate = trained["gates"].get(market_key) if trained else None
                selected_method = gate.get("selected_method", "baseline") if gate else "baseline"
                predicted_price, ml_price = predict_method_price(trained, selected_method, row, current_price, horizon)
                if ml_price is None and selected_method != "baseline":
                    selected_method = "baseline"
                horizon_points[horizon] = predicted_price
                metric_payload = gate or {"selected_method": selected_method, "sample_count": 0}
                radius = float(metric_payload.get("conformal_log_radius", np.log(1.25)))
                lower = float(predicted_price * np.exp(-radius))
                upper = float(predicted_price * np.exp(radius))
                low_ratio, high_ratio = HORIZON_PRICE_CLIP_BOUNDS.get(int(horizon), (0.55, 1.75))
                lower = min(float(predicted_price), max(lower, current_price * low_ratio))
                upper = max(float(predicted_price), min(upper, current_price * high_ratio))
                horizon_intervals[horizon] = (lower, upper)
                market_metrics[grain][market_key][str(horizon)] = metric_payload
                prediction["horizons"][str(horizon)] = {
                    "target_date": (last_ts + pd.Timedelta(days=horizon)).date().isoformat(),
                    "predicted_price": round(float(predicted_price), 2),
                    "confidence_lower": round(lower, 2),
                    "confidence_upper": round(upper, 2),
                    "selected_method": selected_method,
                    "prediction_interval": {
                        "lower": round(lower, 2),
                        "upper": round(upper, 2),
                        "coverage_target": round(1.0 - CONFORMAL_ALPHA, 2),
                        "method": "split_conformal_log_residual",
                    },
                    "metrics": {
                        "mape": as_json_float(metric_payload.get("ml_mape")),
                        "mae": as_json_float(metric_payload.get("ml_mae")),
                        "baseline_mape": as_json_float(metric_payload.get("baseline_mape")),
                        "baseline_mae": as_json_float(metric_payload.get("baseline_mae")),
                        "sample_count": int(metric_payload.get("sample_count", 0)),
                        "method_mapes": metric_payload.get("method_mapes", {}),
                        "method_maes": metric_payload.get("method_maes", {}),
                    },
                    "model_price": round(float(ml_price), 2) if ml_price is not None else None,
                }

            payload = {**info, "prediction": prediction}
            market_predictions[grain][market_key] = payload
            market_forecast_series[grain][market_key] = interpolate_market_series(
                last_date, current_price, horizon_points, horizon_intervals
            )
            market_reasoning[grain][market_key] = {
                "headline": f"{grain} forecast for {info.get('market_name') or 'selected mandi'}",
                "bullets": [
                    "Mandi-level forecast is trained on the selected market's own recent price path when enough history is available.",
                    "Farm-gate prices on the website subtract estimated transport and handling charges from the mandi price.",
                    "If location coordinates are missing for this mandi, the website can still show the mandi forecast but cannot compute true nearest-distance selection.",
                ],
                "source": "mandi_release",
            }

        grain_actuals = market_canonical[market_canonical["grain"].eq(grain) & ~market_canonical["state_name"].eq("All States")].copy()
        grain_actuals["date"] = pd.to_datetime(grain_actuals["date"])
        if not grain_actuals.empty:
            cutoff = grain_actuals["date"].max() - pd.Timedelta(days=MANDI_FORECAST_HISTORY_DAYS)
            grain_actuals = grain_actuals[grain_actuals["date"].ge(cutoff)]
        for market_key, market_df in grain_actuals.groupby("state_name", observed=True):
            market_df = market_df.sort_values("date")
            market_actuals[grain][market_key] = {
                "context": [
                    {
                        "date": date.date().isoformat(),
                        "price": round(float(price), 2),
                        "is_observed": bool(is_observed),
                    }
                    for date, price, is_observed in zip(market_df["date"], market_df["price"], market_df["is_observed"])
                ]
            }

    (RELEASE_DIR / "markets.json").write_text(json.dumps({"markets": market_lookup_payload(market_daily)}, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_predictions.json").write_text(json.dumps(market_predictions, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_forecast_series.json").write_text(json.dumps(market_forecast_series, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_actuals.json").write_text(json.dumps(market_actuals, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_metrics.json").write_text(json.dumps(market_metrics, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_reasoning.json").write_text(json.dumps(market_reasoning, indent=2, allow_nan=False), encoding="utf-8")
    market_canonical.to_csv(RELEASE_DIR / "market_canonical_daily.csv", index=False)
    try:
        market_canonical.to_parquet(RELEASE_DIR / "market_canonical_daily.parquet", index=False)
    except Exception as exc:
        print("Skipping market_canonical_daily.parquet:", exc)
    return market_predictions


def generate_fast_market_predictions(market_daily, market_canonical):
    market_predictions, market_forecast_series, market_actuals, market_metrics, market_reasoning = {}, {}, {}, {}, {}
    if market_daily.empty:
        return market_predictions

    latest_rows = market_daily.sort_values("date").groupby(["grain", "market_key"], as_index=False, observed=True).tail(1)
    lookup = {item["market_key"]: item for item in market_lookup_payload(market_daily)}

    for grain in TARGET_GRAINS:
        market_predictions[grain] = {}
        market_forecast_series[grain] = {}
        market_actuals[grain] = {}
        market_metrics[grain] = {}
        market_reasoning[grain] = {}

        for _, row in latest_rows[latest_rows["grain"].eq(grain)].iterrows():
            market_key = row["market_key"]
            info = lookup.get(market_key, {"market_key": market_key})
            state = row.get("state_name")
            state_prediction = predictions.get(grain, {}).get(state) or predictions.get(grain, {}).get("All States") or {}
            state_current = float(state_prediction.get("current_price") or row["price"] or 0)
            current_price = float(row["price"])
            ratio = current_price / state_current if state_current > 0 else 1.0
            last_ts = pd.to_datetime(row["date"])
            last_date = last_ts.date()
            horizon_points = {}
            horizon_intervals = {}
            prediction = {
                "current_price": round(current_price, 2),
                "last_actual_date": last_date.isoformat(),
                "forecast_start_date": (last_ts + pd.Timedelta(days=1)).date().isoformat(),
                "status": "fresh",
                "model_mode": "market_adjusted_state_model",
                "horizons": {},
            }
            for horizon in HORIZONS:
                state_horizon = (state_prediction.get("horizons") or {}).get(str(horizon), {})
                state_predicted_price = float(state_horizon.get("predicted_price") or state_current or current_price)
                predicted_price = state_predicted_price * ratio
                horizon_points[horizon] = predicted_price
                state_interval = state_horizon.get("prediction_interval") or {}
                lower = float(state_interval.get("lower") or state_predicted_price) * ratio
                upper = float(state_interval.get("upper") or state_predicted_price) * ratio
                if lower > predicted_price:
                    lower = predicted_price
                if upper < predicted_price:
                    upper = predicted_price
                horizon_intervals[horizon] = (lower, upper)
                metrics_payload = dict(state_horizon.get("metrics") or {})
                metrics_payload["market_adjustment_ratio"] = round(float(ratio), 6)
                prediction["horizons"][str(horizon)] = {
                    "target_date": (last_ts + pd.Timedelta(days=horizon)).date().isoformat(),
                    "predicted_price": round(float(predicted_price), 2),
                    "confidence_lower": round(lower, 2),
                    "confidence_upper": round(upper, 2),
                    "selected_method": f"market_adjusted_{state_horizon.get('selected_method') or 'state_model'}",
                    "prediction_interval": {
                        "lower": round(lower, 2),
                        "upper": round(upper, 2),
                        "coverage_target": state_interval.get("coverage_target", round(1.0 - CONFORMAL_ALPHA, 2)),
                        "method": state_interval.get("method", "scaled_state_interval"),
                    },
                    "metrics": metrics_payload,
                    "model_price": round(float(predicted_price), 2),
                }
                market_metrics[grain].setdefault(market_key, {})[str(horizon)] = metrics_payload

            market_predictions[grain][market_key] = {**info, "prediction": prediction}
            market_forecast_series[grain][market_key] = interpolate_market_series(
                last_date, current_price, horizon_points, horizon_intervals
            )
            market_reasoning[grain][market_key] = {
                "headline": f"{grain} forecast for {info.get('market_name') or 'selected mandi'}",
                "bullets": [
                    "This fast mandi forecast anchors on the mandi's latest observed price.",
                    "The 7/30/90 day movement comes from the trained state/national model and is adjusted to this mandi's current price level.",
                    "This fallback is used only if full mandi-level training is disabled or cannot complete within the available runtime.",
                ],
                "source": "market_adjusted_state_model",
            }

        grain_actuals = market_canonical[market_canonical["grain"].eq(grain) & ~market_canonical["state_name"].eq("All States")].copy()
        grain_actuals["date"] = pd.to_datetime(grain_actuals["date"])
        if not grain_actuals.empty:
            cutoff = grain_actuals["date"].max() - pd.Timedelta(days=MANDI_FORECAST_HISTORY_DAYS)
            grain_actuals = grain_actuals[grain_actuals["date"].ge(cutoff)]
        for market_key, market_df in grain_actuals.groupby("state_name", observed=True):
            market_df = market_df.sort_values("date")
            market_actuals[grain][market_key] = {
                "context": [
                    {
                        "date": date.date().isoformat(),
                        "price": round(float(price), 2),
                        "is_observed": bool(is_observed),
                    }
                    for date, price, is_observed in zip(market_df["date"], market_df["price"], market_df["is_observed"])
                ]
            }

    (RELEASE_DIR / "markets.json").write_text(json.dumps({"markets": market_lookup_payload(market_daily)}, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_predictions.json").write_text(json.dumps(market_predictions, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_forecast_series.json").write_text(json.dumps(market_forecast_series, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_actuals.json").write_text(json.dumps(market_actuals, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_metrics.json").write_text(json.dumps(market_metrics, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "market_reasoning.json").write_text(json.dumps(market_reasoning, indent=2, allow_nan=False), encoding="utf-8")
    market_canonical.to_csv(RELEASE_DIR / "market_canonical_daily.csv", index=False)
    try:
        market_canonical.to_parquet(RELEASE_DIR / "market_canonical_daily.parquet", index=False)
    except Exception as exc:
        print("Skipping market_canonical_daily.parquet:", exc)
    return market_predictions


if ENABLE_MANDI_LEVEL_RELEASE:
    print("\nBuilding mandi-level forecast release files...")
    market_daily = load_market_level_sources()
    if market_daily.empty:
        print("No mandi-level source rows found. Writing empty mandi sidecar files.")
        (RELEASE_DIR / "markets.json").write_text(json.dumps({"markets": []}, indent=2), encoding="utf-8")
        for file_name in ["market_predictions.json", "market_forecast_series.json", "market_actuals.json", "market_metrics.json", "market_reasoning.json"]:
            (RELEASE_DIR / file_name).write_text(json.dumps({}, indent=2), encoding="utf-8")
    else:
        market_canonical = build_market_training_canonical(market_daily)
        print(f"Market training canonical rows: {len(market_canonical):,}")
        display(pd.DataFrame(market_lookup_payload(market_daily)).head(20))
        if ENABLE_MANDI_LEVEL_FULL_TRAINING:
            print("Full global mandi-aware retraining is enabled.")
            previous_min_observed_days = MIN_STATE_OBSERVED_DAYS
            previous_training_scope = TRAINING_SCOPE
            MIN_STATE_OBSERVED_DAYS = MIN_MARKET_OBSERVED_DAYS
            TRAINING_SCOPE = "all"
            try:
                market_registry = train_models(market_canonical)
                market_predictions = generate_market_predictions(market_daily, market_canonical, market_registry)
                os.environ["MARKET_MODEL_MODE"] = "global_mandi_aware"
            except Exception as exc:
                print("Full mandi training failed; using market-adjusted state fallback:", exc)
                market_predictions = generate_fast_market_predictions(market_daily, market_canonical)
                os.environ["MARKET_MODEL_MODE"] = "market_adjusted_state_fallback"
            finally:
                MIN_STATE_OBSERVED_DAYS = previous_min_observed_days
                TRAINING_SCOPE = previous_training_scope
        else:
            print("Using fast mandi mode: current mandi price + trained state/national forecast ratios.")
            market_predictions = generate_fast_market_predictions(market_daily, market_canonical)
            os.environ["MARKET_MODEL_MODE"] = "market_adjusted_state_model"
        print(f"Mandi-level predictions written for {sum(len(v) for v in market_predictions.values()):,} grain/mandi slots.")
else:
    print("Mandi-level release disabled with ENABLE_MANDI_LEVEL_RELEASE=false")


## 13. Finalize and Quality-Gate the Production Release Bundle

Required artifacts, freshness, finite prices, forecast swings, interval validity, selected-method superiority, and efficiency payload size are checked before the release is mirrored to `/kaggle/working` and zipped.


In [ ]:
manifest = finalize_release(canonical)
release_files = release_file_summary()
print("\nFinal manifest")
display(pd.DataFrame([manifest]))